# Student Attrition Machine Learning

Sprint 2 machine-learning workflow that predicts twelve-month **student** attrition
(`is_twelve_month_student_attrition`) from the deidentified synthetic Delta tables in
`workspace.student_aggregate`.

This notebook is built in three human-gated phases:

| Phase | Scope | Status |
|---|---|---|
| Phase 1 | Source audit and fact-centred STAR-schema feature assembly | **Approved 2026-08-06** (T1-T8 complete) |
| Phase 2 | Split, preprocessing, Random Forest training, MLflow, evaluation | **Approved 2026-08-06** (T9-T21 complete) |
| Phase 3 | Inference, risk flag generation, Delta persistence, retrieval validation | **Approved 2026-08-06** (T22-T32 complete) |

**Sprint 2 machine-learning implementation is complete.** The approved prediction table
`workspace.student_aggregate.student_attrition_risk_prediction` is written in section 28, and only
if every check in section 27.1 passes. No source table is modified.

### Controlling documents

Behaviour is derived from the approved Spec-Driven Development set: `constitution.md`,
`specification.md`, `plan.md`, `tasks.md`. Tasks **T1-T32** are marked complete. Limitations
**L-1 to L-3** in `plan.md` govern how stored risk percentages should be read.

### Execution notes

- Runs top to bottom on serverless-compatible Databricks compute with access to
  `workspace.student_aggregate`.
- PySpark only. The dataset is never collected to a single pandas DataFrame; only small
  aggregate results are brought to the driver for reporting.
- Validation cells raise `AssertionError` (hard stop) rather than silently repairing data.
- No `cache()` or `persist()`. Serverless compute rejects them with
  `NOT_SUPPORTED_WITH_SERVERLESS: PERSIST TABLE is not supported`. Where caching would normally
  have amortised several counts over one DataFrame, the counts are folded into a single
  aggregate instead, so each check costs one pass rather than one pass per figure. The same
  constraint holds in Phases 2 and 3; where a result must be reused, it is read back from Delta
  rather than cached.
- Serverless caps in-memory models at 100 MB each and 1 GB per session, and Spark Connect holds a
  fitted model on the driver while any Python reference to it survives. Re-running the training
  cell releases the previous fit first. If the session overflows anyway, restart Python and run
  from the top; see section 20.
- Phase 3 applies the model to the full population exactly once. The scored result is materialised
  to a transient staging table, validated there, published to the approved table, and the staging
  table is dropped. Validation still gates the write, so the T27 guarantee is unchanged. See
  section 26 for the reasoning.

**Re-running after a failure?** Run **section 0** first, then **Run all**. Do not re-run only the
training cell.

## 0. Session reset (run this first if you are re-running after a failure)

Run this cell, then **Run all** from the top. It clears cached models and stale MLflow runs. It
does not retrain anything and does not delete any Delta table.

In [0]:
import gc
import sys

import mlflow


# --------------------------------------------------------------------------------------
# 1. End any MLflow run left open by a previous failed execution
# --------------------------------------------------------------------------------------
if mlflow.active_run() is not None:
    stale_run_id = mlflow.active_run().info.run_id
    mlflow.end_run()
    print(f"Ended stale MLflow run: {stale_run_id}")


# --------------------------------------------------------------------------------------
# 2. Remove fitted models, estimators, and derived DataFrames
#
# random_forest and attrition_model are included so a stale estimator or fitted model cannot
# be reused accidentally after RF constants are changed.
# --------------------------------------------------------------------------------------
_MODEL_AND_PIPELINE_NAMES = (
    # Fitted model
    "attrition_model",

    # Unfitted classifier
    "random_forest",

    # Stateless preprocessing objects / metadata
    "vector_assembler",
    "NUMERIC_MEDIANS",
    "CATEGORY_LEVELS",
    "FEATURE_VECTOR_COLUMNS_BY_FEATURE",
    "ASSEMBLER_INPUT_COLUMNS",
    "numeric_median_row",

    # Random Forest parameter record
    "RF_PARAMETERS",

    # Prepared training and evaluation DataFrames
    "train_prepared_df",
    "validation_prepared_df",
    "test_prepared_df",

    # Scoring and persistence DataFrames
    "scored_inference_df",
    "staging_output_df",
    "staged_df",
    "validation_stage_df",
    "prediction_output_df",
    "inference_input_df",
    "persisted_df",

    # Feature-importance objects
    "raw_importances",
    "slot_index",
    "importance_by_feature",
    "importance_df",
    "slots_by_feature",
)

_released = []

for _name in _MODEL_AND_PIPELINE_NAMES:
    if _name in globals():
        del globals()[_name]
        _released.append(_name)


# --------------------------------------------------------------------------------------
# 3. Clear stored exception references
#
# Databricks/IPython may retain the stack frame from the previous failed fit. That frame
# can hold references to Spark Connect objects even after their normal variable is deleted.
# --------------------------------------------------------------------------------------
for _attr in (
    "last_traceback",
    "last_value",
    "last_type",
    "last_exc",
):
    try:
        delattr(sys, _attr)
    except (AttributeError, TypeError):
        pass


# Clear common IPython output references that may retain the previous exception or object.
try:
    _ipython = get_ipython()

    if _ipython is not None:
        for _history_name in ("_", "__", "___"):
            if _history_name in _ipython.user_ns:
                _ipython.user_ns[_history_name] = None
except Exception as _history_cleanup_error:
    print(
        "Note: IPython history references could not be completely cleared: "
        f"{_history_cleanup_error}"
    )


# --------------------------------------------------------------------------------------
# 4. Request Python garbage collection
# --------------------------------------------------------------------------------------
_collected_objects = 0

for _ in range(3):
    _collected_objects += gc.collect()


print("=" * 80)
print("SESSION RESET COMPLETE")
print("=" * 80)
print(f"Released names             : {_released or 'none'}")
print(f"Garbage-collected objects  : {_collected_objects}")
print()
print("Next steps:")
print("  1. Confirm the approved Random Forest constants are saved (100 trees, depth 7).")
print("  2. Run the notebook from the top.")
print("  3. Do not run only the training cell.")
print()
print("For a completely clean Spark Connect Python session, run:")
print("  dbutils.library.restartPython()")
print("Then use Run all from the top.")

## 1. Constants and column roles (T1, T4, T7, T8)

Every source table, the output table, the target, the identifiers, the prohibited leakage
columns, the risk threshold, and the random seed are declared once here. The threshold, seed,
and output table are defined in Phase 1 but are not used until Phases 2 and 3.

In [0]:
from datetime import datetime, timezone

import pyspark.sql.functions as F
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType
)

# ---- Catalog / schema -------------------------------------------------------------------------
CATALOG = "workspace"
SCHEMA = "student_aggregate"
FQ = f"{CATALOG}.{SCHEMA}"

# ---- Authoritative modelling spine -------------------------------------------------------------
FACT_TABLE = f"{FQ}.rpt_student_management__fact__all_enrolment_eftsl__deidentified"

# ---- Dimension tables in the declared source inventory -----------------------------------------
TEACHING_PERIOD_TABLE = f"{FQ}.dwh_learning_and_teaching__teaching_period"
COURSE_TABLE = f"{FQ}.dwh_curriculum__course"
COURSE_OFFERING_TABLE = f"{FQ}.dwh_curriculum__course_offering"
MODULE_TABLE = f"{FQ}.dwh_curriculum__module"
MODULE_OFFERING_TABLE = f"{FQ}.dwh_curriculum__module_offering"
UNIT_TABLE = f"{FQ}.dwh_curriculum__unit"
UNIT_OFFERING_TABLE = f"{FQ}.dwh_curriculum__unit_offering"
THESIS_TABLE = f"{FQ}.dwh_curriculum__thesis"
STUDY_AREA_A_TABLE = f"{FQ}.dwh_curriculum__study_area_a"
STUDY_AREA_B_TABLE = f"{FQ}.dwh_curriculum__study_area_b"
ORGANISATION_TABLE = f"{FQ}.dwh_internal_organisation__mapped_academic_organisation_hierarchy"

# ---- Approved Phase 3 output table (declared only; not written in Phase 1) ----------------------
PREDICTION_TABLE = f"{FQ}.student_attrition_risk_prediction"

# ---- Target, identifiers, prohibited columns ----------------------------------------------------
TARGET_COLUMN = "is_twelve_month_student_attrition"
STUDENT_ID_COLUMN = "student_deidentified_hash"
ENROLMENT_ID_COLUMN = "enrolment_deidentified_hash"

IDENTIFIER_COLUMNS = [STUDENT_ID_COLUMN, ENROLMENT_ID_COLUMN]

# Columns that must never reach the feature vector because they reveal or follow the outcome.
PROHIBITED_LEAKAGE_COLUMNS = [
    TARGET_COLUMN,
    "is_twelve_month_course_attrition",
    "is_tcsi_student_attrition",
    "is_tcsi_course_attrition",
    "is_withdrawn_student_attrition",
    "is_withdrawn_course_attrition",
    "student_attrition_deidentified_hash",
    "course_attrition_deidentified_hash",
]

# ---- Approved model settings (declared in Phase 1, applied in Phases 2 and 3) --------------------
RANDOM_SEED = 42
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
RISK_THRESHOLD = 0.50

PHASE_1_RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat(timespec="seconds")

print("Phase 1 configuration")
print(f"  fact table        : {FACT_TABLE}")
print(f"  target            : {TARGET_COLUMN}")
print(f"  student identifier: {STUDENT_ID_COLUMN}")
print(f"  leakage exclusions: {PROHIBITED_LEAKAGE_COLUMNS}")
print(f"  seed / split      : {RANDOM_SEED} / {TRAIN_FRACTION}-{VALIDATION_FRACTION}-{TEST_FRACTION}")
print(f"  risk threshold    : {RISK_THRESHOLD}")
print(f"  Phase 3 output    : {PREDICTION_TABLE} (not written in Phase 1)")
print(f"  run timestamp     : {PHASE_1_RUN_TIMESTAMP}Z")

## 2. Profiling helpers

Small, reusable helpers used by the source audit. Each helper returns driver-side Python
values built from Spark aggregates, so no large dataset is materialised locally.

In [0]:
def table_exists(fqn: str) -> bool:
    """Return True when the table can be resolved in Unity Catalog."""
    try:
        return spark.catalog.tableExists(fqn)
    except Exception:
        return False


def non_null_counts(df) -> dict:
    """Non-null count for every column, computed in a single pass."""
    if not df.columns:
        return {}
    aggregates = [F.count(F.col(f"`{c}`")).alias(c) for c in df.columns]
    return df.agg(*aggregates).collect()[0].asDict()


def key_quality(df, key_column: str) -> dict:
    """Null-key count, distinct-key count, and duplicated-key-value count for a candidate key."""
    if key_column not in df.columns:
        return {"key_present": False}
    null_keys = df.filter(F.col(key_column).isNull()).count()
    distinct_keys = df.select(key_column).distinct().count()
    duplicated_key_values = (
        df.groupBy(key_column).count().filter(F.col("count") > 1).count()
    )
    return {
        "key_present": True,
        "null_keys": null_keys,
        "distinct_keys": distinct_keys,
        "duplicated_key_values": duplicated_key_values,
    }


def column_type_map(df) -> dict:
    return {field.name: field.dataType.simpleString() for field in df.schema.fields}


def profile_table(fqn: str, candidate_key: str, fact_relationship: str, candidate_attributes: list) -> dict:
    """Profile one source table against the Phase 1 source-inventory requirements."""
    profile = {
        "table": fqn,
        "candidate_key": candidate_key,
        "fact_relationship": fact_relationship,
        "declared_candidate_attributes": candidate_attributes,
        "exists": False,
        "readable": False,
        "error": None,
        "row_count": None,
        "column_count": None,
        "populated_columns": [],
        "all_null_columns": [],
        "types": {},
    }

    if not table_exists(fqn):
        profile["error"] = "table not found in Unity Catalog"
        return profile
    profile["exists"] = True

    try:
        df = spark.table(fqn)
        profile["readable"] = True
    except Exception as exc:  # readability is part of the required evidence
        profile["error"] = str(exc)
        return profile

    profile["types"] = column_type_map(df)
    profile["column_count"] = len(df.columns)
    profile["row_count"] = df.count()

    counts = non_null_counts(df)
    profile["populated_columns"] = sorted([c for c, v in counts.items() if v and v > 0])
    profile["all_null_columns"] = sorted([c for c, v in counts.items() if not v])
    profile["non_null_counts"] = counts
    profile.update(key_quality(df, candidate_key))

    # Which of the declared candidate attributes actually carry data in the current generation.
    profile["candidate_attributes_populated"] = [
        c for c in candidate_attributes if counts.get(c, 0) and counts.get(c, 0) > 0
    ]
    profile["candidate_attributes_all_null"] = [
        c for c in candidate_attributes if c in counts and not counts.get(c)
    ]
    profile["candidate_attributes_missing"] = [c for c in candidate_attributes if c not in counts]
    return profile

## 3. Source inventory - profile the 12 generated Delta tables

For each table the audit records existence, readability, schema, row count, candidate business
key, null-key count, duplicate-key count, populated versus all-null columns, the possible
relationship to the fact table, and the candidate prediction-time-valid attributes.

The declared candidate attributes below are the columns that *could* be used if populated; the
profile reports which of them actually carry data in the current synthetic generation.

In [0]:
SOURCE_INVENTORY = [
    {
        "fqn": FACT_TABLE,
        "candidate_key": ENROLMENT_ID_COLUMN,
        "fact_relationship": "authoritative modelling spine (self)",
        "candidate_attributes": [
            "age_at_census",
            "age_band",
            "socioeconomic_status",
            "regional_remote_status",
            "student_gender",
            "student_is_international_student",
            "is_international_student",
            "international_domestic_student",
            "international_domestic_enrolment",
            "student_is_first_nations_student",
            "attendance_mode",
            "course_admission_load_category",
            "eftsl",
            "enrolment_year",
            "commencing_continuing",
            "commencing_continuing_period",
            "is_commencing",
            "cumulative_credit_points_enrolled",
            "cumulative_credit_points_passed",
            "cumulative_credit_points_failed",
            "cumulative_credit_points_withdrawn",
            "census_date",
        ],
    },
    {
        "fqn": TEACHING_PERIOD_TABLE,
        "candidate_key": "teaching_period_key_hash",
        "fact_relationship": "fact.teaching_period_key_hash -> teaching_period.teaching_period_key_hash",
        "candidate_attributes": ["teaching_period", "teaching_period_code", "calendar_year", "census_date"],
    },
    {
        "fqn": COURSE_TABLE,
        "candidate_key": "course_key_hash",
        "fact_relationship": "fact.course_key_hash -> course.course_key_hash",
        "candidate_attributes": [
            "course_group",
            "course_level",
            "broad_primary_field_of_education",
            "narrow_primary_field_of_education",
            "detailed_primary_field_of_education",
        ],
    },
    {
        "fqn": COURSE_OFFERING_TABLE,
        "candidate_key": "course_offering_key_hash",
        "fact_relationship": "fact.course_offering_key_hash -> course_offering.course_offering_key_hash",
        "candidate_attributes": ["location", "study_period", "year", "available_to_students"],
    },
    {
        "fqn": MODULE_TABLE,
        "candidate_key": "module_key_hash",
        "fact_relationship": "no verified single-target relationship from the fact table",
        "candidate_attributes": ["module_type", "broad_field_of_education"],
    },
    {
        "fqn": MODULE_OFFERING_TABLE,
        "candidate_key": "module_offering_key_hash",
        "fact_relationship": "no verified single-target relationship from the fact table",
        "candidate_attributes": ["location", "study_period"],
    },
    {
        "fqn": UNIT_TABLE,
        "candidate_key": "unit_key_hash",
        "fact_relationship": "fact.curriculum_item_key_hash -> unit.unit_key_hash",
        "candidate_attributes": ["unit_type", "broad_field_of_education", "narrow_field_of_education"],
    },
    {
        "fqn": UNIT_OFFERING_TABLE,
        "candidate_key": "unit_offering_key_hash",
        "fact_relationship": "fact.curriculum_item_offering_key_hash -> unit_offering.unit_offering_key_hash",
        "candidate_attributes": ["location", "study_period", "year"],
    },
    {
        "fqn": THESIS_TABLE,
        "candidate_key": "thesis_key_hash",
        "fact_relationship": "no verified single-target relationship from the fact table",
        "candidate_attributes": ["thesis_type", "broad_field_of_education"],
    },
    {
        "fqn": STUDY_AREA_A_TABLE,
        "candidate_key": "study_area_a_key_hash",
        "fact_relationship": "fact.study_area_a_key_hash -> study_area_a.study_area_a_key_hash",
        "candidate_attributes": ["study_area_a_type", "broad_field_of_education"],
    },
    {
        "fqn": STUDY_AREA_B_TABLE,
        "candidate_key": "study_area_b_key_hash",
        "fact_relationship": "fact.study_area_b_key_hash -> study_area_b.study_area_b_key_hash",
        "candidate_attributes": ["study_area_b_type", "broad_field_of_education"],
    },
    {
        "fqn": ORGANISATION_TABLE,
        "candidate_key": "academic_organisation_hierarchy_key_hash",
        "fact_relationship": (
            "fact.academic_organisation_hierarchy_key_hash -> "
            "organisation.academic_organisation_hierarchy_key_hash"
        ),
        "candidate_attributes": [
            "organisation_type",
            "faculty_organisation_name",
            "school_organisation_name",
            "organisation_level",
        ],
    },
]

SOURCE_PROFILES = {}
for entry in SOURCE_INVENTORY:
    SOURCE_PROFILES[entry["fqn"]] = profile_table(
        entry["fqn"], entry["candidate_key"], entry["fact_relationship"], entry["candidate_attributes"]
    )
    print(f"profiled: {entry['fqn']}")

### 3.1 Source-inventory summary

In [0]:
def _as_long(value):
    return None if value is None else int(value)


profile_summary_rows = [
    (
        p["table"].split(".")[-1],
        bool(p["exists"]),
        bool(p["readable"]),
        _as_long(p["row_count"]),
        _as_long(p["column_count"]),
        _as_long(len(p["populated_columns"])),
        _as_long(len(p["all_null_columns"])),
        p["candidate_key"],
        _as_long(p.get("null_keys")),
        _as_long(p.get("distinct_keys")),
        _as_long(p.get("duplicated_key_values")),
        p["fact_relationship"],
        ", ".join(p.get("candidate_attributes_populated", [])) or "(none populated)",
        p.get("error") or "",
    )
    for p in SOURCE_PROFILES.values()
]

source_summary_schema = StructType([
    StructField("table", StringType(), False),
    StructField("exists", BooleanType(), False),
    StructField("readable", BooleanType(), False),
    StructField("row_count", LongType(), True),
    StructField("column_count", LongType(), True),
    StructField("populated_columns", LongType(), True),
    StructField("all_null_columns", LongType(), True),
    StructField("candidate_key", StringType(), True),
    StructField("null_keys", LongType(), True),
    StructField("distinct_keys", LongType(), True),
    StructField("duplicated_key_values", LongType(), True),
    StructField("possible_fact_relationship", StringType(), True),
    StructField("prediction_time_candidate_attributes_populated", StringType(), True),
    StructField("error", StringType(), True),
])

source_summary_df = spark.createDataFrame(profile_summary_rows, schema=source_summary_schema)
display(source_summary_df)

# Hard stop if any declared source table is missing or unreadable.
unusable_sources = [p["table"] for p in SOURCE_PROFILES.values() if not (p["exists"] and p["readable"])]
assert not unusable_sources, f"Phase 1 stop: source tables missing or unreadable -> {unusable_sources}"
print("All 12 declared source tables exist and are readable.")

### 3.2 Column-level populated / all-null detail

Full evidence for the "populated and all-null columns" requirement. Any dimension whose
non-key columns are entirely NULL cannot contribute a feature and is excluded from joining.

In [0]:
column_detail_rows = []
for profile in SOURCE_PROFILES.values():
    short_name = profile["table"].split(".")[-1]
    for column, non_null in sorted(profile.get("non_null_counts", {}).items()):
        row_count = profile["row_count"] or 0
        column_detail_rows.append(
            (
                short_name,
                column,
                profile["types"].get(column),
                int(non_null or 0),
                round(100.0 * (1 - (non_null or 0) / row_count), 4) if row_count else None,
                bool(non_null),
            )
        )

column_detail_df = spark.createDataFrame(
    column_detail_rows,
    schema=StructType([
        StructField("table", StringType(), False),
        StructField("column", StringType(), False),
        StructField("data_type", StringType(), True),
        StructField("non_null_rows", LongType(), True),
        StructField("null_percentage", DoubleType(), True),
        StructField("is_populated", BooleanType(), True),
    ]),
)
display(column_detail_df.orderBy("table", "column"))

## 4. Fact-grain validation (T2, T5)

The fact table is the authoritative spine: every candidate modelling row originates here.

If more than one eligible fact row exists per student the notebook **stops**. Deduplication,
`distinct()`, first-row selection, arbitrary aggregation, and latest-row assumptions are not
applied without human approval.

In [0]:
fact_df = spark.table(FACT_TABLE)

# Serverless compute rejects cache()/persist(), so every action re-reads the table. The row-level
# checks below are therefore folded into one aggregate; only the per-student duplicate count, which
# needs a shuffle, is a second pass.
grain_stats = fact_df.agg(
    F.count(F.lit(1)).alias("fact_row_count"),
    F.count(F.when(F.col(STUDENT_ID_COLUMN).isNull(), F.lit(1))).alias("null_student_hash"),
    F.count(F.when(F.col(ENROLMENT_ID_COLUMN).isNull(), F.lit(1))).alias("null_enrolment_hash"),
    F.countDistinct(F.col(STUDENT_ID_COLUMN)).alias("distinct_students"),
    F.countDistinct(F.col(ENROLMENT_ID_COLUMN)).alias("distinct_enrolments"),
    F.count(F.when(F.col(TARGET_COLUMN).isNull(), F.lit(1))).alias("null_target"),
).collect()[0]

fact_row_count = _as_long(grain_stats["fact_row_count"])
null_student_hash = _as_long(grain_stats["null_student_hash"])
null_enrolment_hash = _as_long(grain_stats["null_enrolment_hash"])
distinct_students = _as_long(grain_stats["distinct_students"])
distinct_enrolments = _as_long(grain_stats["distinct_enrolments"])
null_target = _as_long(grain_stats["null_target"])
students_with_multiple_rows = (
    fact_df.groupBy(STUDENT_ID_COLUMN).count().filter(F.col("count") > 1).count()
)

fact_grain_df = spark.createDataFrame(
    [
        ("fact_row_count", fact_row_count),
        ("null_student_deidentified_hash", null_student_hash),
        ("null_enrolment_deidentified_hash", null_enrolment_hash),
        ("distinct_students", distinct_students),
        ("distinct_enrolments", distinct_enrolments),
        (f"null_{TARGET_COLUMN}", null_target),
        ("students_with_more_than_one_fact_row", students_with_multiple_rows),
    ],
    schema=["check", "value"],
)
display(fact_grain_df)

assert null_student_hash == 0, (
    f"Phase 1 stop: {null_student_hash} fact rows have a NULL {STUDENT_ID_COLUMN}."
)
assert null_enrolment_hash == 0, (
    f"Phase 1 stop: {null_enrolment_hash} fact rows have a NULL {ENROLMENT_ID_COLUMN}."
)
assert students_with_multiple_rows == 0, (
    "Phase 1 stop: the fact table contains more than one row for "
    f"{students_with_multiple_rows} students. The approved modelling grain is one row per "
    f"{STUDENT_ID_COLUMN}. Resolving this requires an approved, documented eligibility rule "
    "(for example is_latest_course_enrolment_with_study_load or is_major_course_for_census_date); "
    "deduplication must not be applied without human approval."
)
print(
    f"Fact grain confirmed: {fact_row_count:,} rows, {distinct_students:,} distinct students, "
    "one eligible fact row per student."
)

### 4.1 Target availability and class distribution (T4, T6)

In [0]:
target_values_df = (
    fact_df.groupBy(F.col(TARGET_COLUMN).alias("target_value"))
    .count()
    .withColumn("percentage", F.round(100.0 * F.col("count") / F.lit(fact_row_count), 4))
    .orderBy("target_value")
)
display(target_values_df)

observed_target_values = {row["target_value"] for row in target_values_df.collect()}
allowed_target_values = {True, False, None}
assert observed_target_values.issubset(allowed_target_values), (
    f"Phase 1 stop: {TARGET_COLUMN} contains non-binary values -> "
    f"{observed_target_values - allowed_target_values}"
)
print(f"Target {TARGET_COLUMN} contains only binary values. Observed: {sorted(str(v) for v in observed_target_values)}")

## 5. Join map verification (T3)

Every expected fact-to-dimension relationship is verified against the actual schemas before any
join is performed: both columns must exist, types must be compatible, the dimension key must be
unique, and matched / unmatched key counts are recorded.

Verification is performed for **all** expected relationships, including those that are ultimately
not joined, so the reviewer can see the evidence behind each inclusion or exclusion decision.

In [0]:
EXPECTED_RELATIONSHIPS = [
    ("course", "course_key_hash", COURSE_TABLE, "course_key_hash"),
    ("course_offering", "course_offering_key_hash", COURSE_OFFERING_TABLE, "course_offering_key_hash"),
    ("unit", "curriculum_item_key_hash", UNIT_TABLE, "unit_key_hash"),
    ("unit_offering", "curriculum_item_offering_key_hash", UNIT_OFFERING_TABLE, "unit_offering_key_hash"),
    ("study_area_a", "study_area_a_key_hash", STUDY_AREA_A_TABLE, "study_area_a_key_hash"),
    ("study_area_b", "study_area_b_key_hash", STUDY_AREA_B_TABLE, "study_area_b_key_hash"),
    (
        "organisation",
        "academic_organisation_hierarchy_key_hash",
        ORGANISATION_TABLE,
        "academic_organisation_hierarchy_key_hash",
    ),
    ("teaching_period", "teaching_period_key_hash", TEACHING_PERIOD_TABLE, "teaching_period_key_hash"),
    # The Power BI relationship file also maps curriculum_item_key_hash to module and thesis.
    # Both are verified here so the ambiguity is measured rather than assumed.
    ("module (ambiguous)", "curriculum_item_key_hash", MODULE_TABLE, "module_key_hash"),
    ("thesis (ambiguous)", "curriculum_item_key_hash", THESIS_TABLE, "thesis_key_hash"),
    (
        "module_offering (ambiguous)",
        "curriculum_item_offering_key_hash",
        MODULE_OFFERING_TABLE,
        "module_offering_key_hash",
    ),
]

fact_types = column_type_map(fact_df)
relationship_rows = []

for alias, fact_key, dim_fqn, dim_key in EXPECTED_RELATIONSHIPS:
    dim_profile = SOURCE_PROFILES[dim_fqn]
    dim_df = spark.table(dim_fqn)
    dim_types = column_type_map(dim_df)

    fact_key_exists = fact_key in fact_types
    dim_key_exists = dim_key in dim_types
    types_compatible = fact_key_exists and dim_key_exists and fact_types[fact_key] == dim_types[dim_key]

    matched = unmatched = None
    dim_key_unique = None
    if fact_key_exists and dim_key_exists:
        dim_keys = dim_df.select(F.col(dim_key).alias("__k")).filter(F.col("__k").isNotNull())
        dim_key_unique = dim_keys.count() == dim_keys.distinct().count()
        matched = fact_df.join(
            dim_keys.distinct(), fact_df[fact_key] == F.col("__k"), "left_semi"
        ).count()
        unmatched = fact_row_count - matched

    relationship_rows.append(
        (
            alias,
            fact_key,
            dim_fqn.split(".")[-1],
            dim_key,
            fact_key_exists,
            dim_key_exists,
            fact_types.get(fact_key),
            dim_types.get(dim_key),
            types_compatible,
            dim_key_unique,
            _as_long(matched),
            _as_long(unmatched),
            round(100.0 * matched / fact_row_count, 4) if matched is not None and fact_row_count else None,
            _as_long(
                len([
                    c for c in dim_profile["populated_columns"]
                    if not c.startswith("__")
                    and not c.endswith("_key")
                    and not c.endswith("_key_hash")
                ])
            ),
        )
    )

relationship_df = spark.createDataFrame(
    relationship_rows,
    schema=StructType([
        StructField("relationship", StringType(), False),
        StructField("fact_column", StringType(), False),
        StructField("dimension_table", StringType(), False),
        StructField("dimension_column", StringType(), False),
        StructField("fact_column_exists", BooleanType(), True),
        StructField("dimension_column_exists", BooleanType(), True),
        StructField("fact_column_type", StringType(), True),
        StructField("dimension_column_type", StringType(), True),
        StructField("types_compatible", BooleanType(), True),
        StructField("dimension_key_unique", BooleanType(), True),
        StructField("matched_fact_rows", LongType(), True),
        StructField("unmatched_fact_rows", LongType(), True),
        StructField("matched_percentage", DoubleType(), True),
        StructField("populated_non_key_dimension_columns", LongType(), True),
    ]),
)
display(relationship_df)

## 6. Join contract

A single guarded helper performs every accepted dimension join. It enforces the full Phase 1
join contract and raises rather than repairing:

1. both join columns exist with compatible types;
2. the dimension key is unique and non-null;
3. only the join key and explicitly approved attributes are selected (no `select("*")`);
4. selected attributes are renamed to source-qualified staging names before joining;
5. a left join is performed from the fact-centred DataFrame;
6. row count is compared before and after;
7. distinct student count is compared before and after;
8. matched and unmatched key counts are recorded;
9. no duplicate column names are introduced.

Any row multiplication, changed student count, duplicate key, or ambiguity stops Phase 1.

In [0]:
JOIN_LEDGER = []


def contracted_left_join(base_df, base_key: str, dim_fqn: str, dim_key: str, attribute_map: dict, alias: str):
    """Left join an approved dimension onto the fact-centred DataFrame under the Phase 1 join contract."""
    dim_df = spark.table(dim_fqn)

    base_types = column_type_map(base_df)
    dim_types = column_type_map(dim_df)

    # (1) columns exist with compatible types
    assert base_key in base_types, f"Join stop [{alias}]: fact column {base_key} not found."
    assert dim_key in dim_types, f"Join stop [{alias}]: dimension column {dim_key} not found."
    assert base_types[base_key] == dim_types[dim_key], (
        f"Join stop [{alias}]: incompatible join types "
        f"{base_key}:{base_types[base_key]} vs {dim_key}:{dim_types[dim_key]}."
    )
    for source_column in attribute_map:
        assert source_column in dim_types, (
            f"Join stop [{alias}]: approved attribute {source_column} not present in {dim_fqn}."
        )

    # (2) dimension key is unique and non-null. Serverless has no cache(), so the three figures are
    # taken in a single aggregate rather than three separate scans of the dimension.
    dim_stats = dim_df.agg(
        F.count(F.lit(1)).alias("rows"),
        F.countDistinct(F.col(dim_key)).alias("distinct_keys"),
        F.count(F.when(F.col(dim_key).isNull(), F.lit(1))).alias("null_keys"),
    ).collect()[0]
    dim_rows = _as_long(dim_stats["rows"])
    dim_distinct_keys = _as_long(dim_stats["distinct_keys"])
    dim_key_nulls = _as_long(dim_stats["null_keys"])
    assert dim_key_nulls == 0, f"Join stop [{alias}]: {dim_key_nulls} NULL keys in {dim_fqn}."
    assert dim_rows == dim_distinct_keys, (
        f"Join stop [{alias}]: {dim_key} is not unique in {dim_fqn} "
        f"({dim_rows} rows vs {dim_distinct_keys} distinct keys)."
    )

    # (3, 4) explicit projection with source-qualified staging names
    join_key_alias = f"__join_key_{alias}"
    projection = [F.col(dim_key).alias(join_key_alias)] + [
        F.col(src).alias(staging) for src, staging in attribute_map.items()
    ]
    dim_selected = dim_df.select(*projection)

    before_stats = base_df.agg(
        F.count(F.lit(1)).alias("rows"),
        F.countDistinct(F.col(STUDENT_ID_COLUMN)).alias("students"),
    ).collect()[0]
    before_rows = _as_long(before_stats["rows"])
    before_students = _as_long(before_stats["students"])

    # (8) matched / unmatched key counts
    matched_rows = base_df.join(
        dim_selected.select(join_key_alias).distinct(),
        base_df[base_key] == F.col(join_key_alias),
        "left_semi",
    ).count()
    unmatched_rows = before_rows - matched_rows

    # (5) left join from the fact-centred DataFrame
    joined = base_df.join(
        dim_selected, base_df[base_key] == dim_selected[join_key_alias], "left"
    ).drop(join_key_alias)

    # (9) no duplicate column names
    duplicate_columns = sorted({c for c in joined.columns if joined.columns.count(c) > 1})
    assert not duplicate_columns, f"Join stop [{alias}]: duplicate column names introduced -> {duplicate_columns}"

    after_stats = joined.agg(
        F.count(F.lit(1)).alias("rows"),
        F.countDistinct(F.col(STUDENT_ID_COLUMN)).alias("students"),
    ).collect()[0]
    after_rows = _as_long(after_stats["rows"])
    after_students = _as_long(after_stats["students"])

    # (6, 7) grain preservation
    assert after_rows == before_rows, (
        f"Join stop [{alias}]: row multiplication detected ({before_rows} -> {after_rows})."
    )
    assert after_students == before_students, (
        f"Join stop [{alias}]: distinct student count changed ({before_students} -> {after_students})."
    )

    JOIN_LEDGER.append(
        {
            "alias": alias,
            "dimension_table": dim_fqn,
            "fact_key": base_key,
            "dimension_key": dim_key,
            "attributes_added": list(attribute_map.values()),
            "rows_before": before_rows,
            "rows_after": after_rows,
            "students_before": before_students,
            "students_after": after_students,
            "matched_rows": matched_rows,
            "unmatched_rows": unmatched_rows,
        }
    )
    print(
        f"join accepted [{alias}]: rows {before_rows:,} -> {after_rows:,}, "
        f"students {before_students:,} -> {after_students:,}, "
        f"matched {matched_rows:,} ({100.0 * matched_rows / before_rows:.2f}%), unmatched {unmatched_rows:,}"
    )
    return joined

## 7. Canonical feature ownership and the feature manifest

Each feature concept has exactly one canonical source. Equivalent attributes that appear in
several dimensions (field of education in course, unit, study area A and study area B) are taken
from the single owning dimension only.

Ownership applied here:

| Concept | Canonical owner |
|---|---|
| Demographics and student status | fact table |
| EFTSL and cumulative credit points | fact table |
| Commencing / continuing status | fact table |
| Teaching-period and calendar attributes | teaching-period dimension |
| Course group, course level, primary field of education | course dimension |
| Organisation attributes | organisation dimension (excluded - no populated attributes) |
| Offering-specific attributes | offering dimensions (excluded - no populated attributes) |

In [0]:
# Each manifest entry: final feature name, source table, source column, source-qualified staging
# name, role (numeric / categorical), prediction-time availability, include/exclude, and reason.
FEATURE_MANIFEST = [
    # ---- Fact-owned demographics and student status ----------------------------------------------
    dict(feature="age_at_census", source_table="fact", source_column="age_at_census",
         staging_name="fact__age_at_census", role="numeric", prediction_time_valid=True,
         decision="include", reason="Student age at the census-date snapshot."),
    dict(feature="socioeconomic_status", source_table="fact", source_column="socioeconomic_status",
         staging_name="fact__socioeconomic_status", role="categorical", prediction_time_valid=True,
         decision="include", reason="SEIFA quartile recorded at census date."),
    dict(feature="regional_remote_status", source_table="fact", source_column="regional_remote_status",
         staging_name="fact__regional_remote_status", role="categorical", prediction_time_valid=True,
         decision="include", reason="ASGS remoteness recorded at census date."),
    dict(feature="student_gender", source_table="fact", source_column="student_gender",
         staging_name="fact__student_gender", role="categorical", prediction_time_valid=True,
         decision="include", reason="Student gender."),
    dict(feature="student_is_international_student", source_table="fact",
         source_column="student_is_international_student",
         staging_name="fact__student_is_international_student", role="categorical",
         prediction_time_valid=True, decision="include",
         reason="Canonical international status. Boolean; cast to string during categorical preprocessing in Phase 2.”"),
    dict(feature="student_is_first_nations_student", source_table="fact",
         source_column="student_is_first_nations_student",
         staging_name="fact__student_is_first_nations_student", role="categorical",
         prediction_time_valid=True, decision="include",
         reason="First Nations status. Boolean; cast to string during categorical preprocessing in Phase 2.”),

    # ---- Fact-owned enrolment and load -----------------------------------------------------------
    dict(feature="attendance_mode", source_table="fact", source_column="attendance_mode",
         staging_name="fact__attendance_mode", role="categorical", prediction_time_valid=True,
         decision="include", reason="Attendance mode group at the enrolment snapshot."),
    dict(feature="eftsl", source_table="fact", source_column="eftsl",
         staging_name="fact__eftsl", role="numeric", prediction_time_valid=True,
         decision="include", reason="Equivalent full-time student load for the enrolment."),
    dict(feature="commencing_continuing", source_table="fact", source_column="commencing_continuing",
         staging_name="fact__commencing_continuing", role="categorical", prediction_time_valid=True,
         decision="include", reason="Canonical commencing/continuing concept for the calendar year."),
    dict(feature="commencing_continuing_period", source_table="fact",
         source_column="commencing_continuing_period",
         staging_name="fact__commencing_continuing_period", role="categorical",
         prediction_time_valid=True, decision="include",
         reason="Approved addition 2026-08-06. Teaching-period commencing status, drawn independently "
                "of the calendar-year value and differing from it on 36% of rows, so it is distinct "
                "information rather than a duplicate encoding."),
    dict(feature="course_admission_load_category", source_table="fact",
         source_column="course_admission_load_category",
         staging_name="fact__course_admission_load_category", role="categorical",
         prediction_time_valid=True, decision="include",
         reason="Approved addition 2026-08-06. Full-time/part-time course-admission load; a distinct "
                "concept from attendance_mode and fully populated."),
    dict(feature="enrolment_year", source_table="fact", source_column="enrolment_year",
         staging_name="fact__enrolment_year", role="numeric", prediction_time_valid=True,
         decision="include", reason="Enrolment year derived from the census date."),

    # ---- Fact-owned cumulative academic outcomes -------------------------------------------------
    dict(feature="cumulative_credit_points_enrolled", source_table="fact",
         source_column="cumulative_credit_points_enrolled",
         staging_name="fact__cumulative_credit_points_enrolled", role="numeric",
         prediction_time_valid=True, decision="include", reason="Cumulative load up to census date."),
    dict(feature="cumulative_credit_points_passed", source_table="fact",
         source_column="cumulative_credit_points_passed",
         staging_name="fact__cumulative_credit_points_passed", role="numeric",
         prediction_time_valid=True, decision="include", reason="Cumulative passes up to census date."),
    dict(feature="cumulative_credit_points_failed", source_table="fact",
         source_column="cumulative_credit_points_failed",
         staging_name="fact__cumulative_credit_points_failed", role="numeric",
         prediction_time_valid=True, decision="include", reason="Cumulative failures up to census date."),
    dict(feature="cumulative_credit_points_withdrawn", source_table="fact",
         source_column="cumulative_credit_points_withdrawn",
         staging_name="fact__cumulative_credit_points_withdrawn", role="numeric",
         prediction_time_valid=True, decision="include", reason="Cumulative withdrawals up to census date."),

    # ---- Course dimension (canonical owner of course group and field of education) ----------------
    dict(feature="course_group", source_table="dwh_curriculum__course", source_column="course_group",
         staging_name="course__course_group", role="categorical", prediction_time_valid=True,
         decision="include", reason="Course group owned by the course dimension."),
    dict(feature="broad_primary_field_of_education", source_table="dwh_curriculum__course",
         source_column="broad_primary_field_of_education",
         staging_name="course__broad_primary_field_of_education", role="categorical",
         prediction_time_valid=True, decision="include",
         reason="Canonical broad field of education; equivalent columns in other dimensions excluded."),
    dict(feature="narrow_primary_field_of_education", source_table="dwh_curriculum__course",
         source_column="narrow_primary_field_of_education",
         staging_name="course__narrow_primary_field_of_education", role="categorical",
         prediction_time_valid=True, decision="include",
         reason="Canonical narrow field of education."),
    dict(feature="detailed_primary_field_of_education", source_table="dwh_curriculum__course",
         source_column="detailed_primary_field_of_education",
         staging_name="course__detailed_primary_field_of_education", role="categorical",
         prediction_time_valid=True, decision="include",
         reason="Canonical detailed field of education. Cardinality reported below for review."),

    # ---- Teaching-period dimension (canonical owner of calendar attributes) -----------------------
    dict(feature="teaching_period", source_table="dwh_learning_and_teaching__teaching_period",
         source_column="teaching_period", staging_name="teaching_period__teaching_period",
         role="categorical", prediction_time_valid=True, decision="include",
         reason="Teaching period owned by the teaching-period dimension."),

    # ---- Target and identifiers (retained, never features) ---------------------------------------
    dict(feature=TARGET_COLUMN, source_table="fact", source_column=TARGET_COLUMN,
         staging_name=TARGET_COLUMN, role="label", prediction_time_valid=False, decision="target",
         reason="Sole supervised label."),
    dict(feature=STUDENT_ID_COLUMN, source_table="fact", source_column=STUDENT_ID_COLUMN,
         staging_name=STUDENT_ID_COLUMN, role="identifier", prediction_time_valid=True,
         decision="identifier", reason="Logical primary key for traceability and application retrieval."),
    dict(feature=ENROLMENT_ID_COLUMN, source_table="fact", source_column=ENROLMENT_ID_COLUMN,
         staging_name=ENROLMENT_ID_COLUMN, role="identifier", prediction_time_valid=True,
         decision="identifier", reason="Retained for traceability only."),

    # ---- Explicit exclusions ---------------------------------------------------------------------
    dict(feature="is_twelve_month_course_attrition", source_table="fact",
         source_column="is_twelve_month_course_attrition", staging_name="-", role="-",
         prediction_time_valid=False, decision="exclude",
         reason="Prohibited leakage column named in the constitution and specification."),
    dict(feature="is_tcsi_student_attrition / is_tcsi_course_attrition / is_withdrawn_*_attrition",
         source_table="fact", source_column="(several)", staging_name="-", role="-",
         prediction_time_valid=False, decision="exclude",
         reason="Post-outcome attrition indicators; also all-NULL in the current generation."),
    dict(feature="student_attrition_deidentified_hash / course_attrition_deidentified_hash",
         source_table="fact", source_column="(several)", staging_name="-", role="-",
         prediction_time_valid=False, decision="exclude",
         reason="Outcome-derived hashes; identifiers are never predictors."),
    dict(feature="all *_key and *_key_hash columns", source_table="fact and dimensions",
         source_column="(several)", staging_name="-", role="-", prediction_time_valid=True,
         decision="exclude", reason="Raw keys and hashes are used for joins and traceability only."),
    dict(feature="age_band", source_table="fact", source_column="age_band", staging_name="-", role="-",
         prediction_time_valid=True, decision="exclude",
         reason="Banded duplicate of age_at_census; the numeric form is canonical."),
    dict(feature="is_international_student / international_domestic_enrolment / international_domestic_student",
         source_table="fact", source_column="(several)", staging_name="-", role="-",
         prediction_time_valid=True, decision="exclude",
         reason="Exact duplicates of the canonical student_is_international_student."),
    dict(feature="is_commencing / is_commencing_12m / is_commencing_period / is_commencing_half_year / "
                 "commencing_continuing_12m / commencing_continuing_half_year",
         source_table="fact", source_column="(several)", staging_name="-", role="-",
         prediction_time_valid=True, decision="exclude",
         reason="Boolean or duplicate encodings of the two retained commencing features; "
                "commencing_continuing_12m equals commencing_continuing on 100% of rows. "
                "Exclusion confirmed by human review 2026-08-06."),
    dict(feature="is_enrolment / is_study_load", source_table="fact", source_column="(several)",
         staging_name="-", role="-", prediction_time_valid=True, decision="exclude",
         reason="Populated but single-valued (constant true), so they carry zero variance. "
                "Exclusion confirmed by human review 2026-08-06."),
    dict(feature="census_date / course_enrolment_census_date", source_table="fact",
         source_column="(several)", staging_name="-", role="-", prediction_time_valid=True,
         decision="exclude", reason="Snapshot reference dates, not approved predictive features; "
                                    "enrolment_year already carries the calendar signal."),
    dict(feature="course_level", source_table="dwh_curriculum__course", source_column="course_level",
         staging_name="-", role="-", prediction_time_valid=True, decision="exclude",
         reason="Populated with the same values as course_group in the current generation."),
    dict(feature="calendar_year", source_table="dwh_learning_and_teaching__teaching_period",
         source_column="calendar_year", staging_name="-", role="-", prediction_time_valid=True,
         decision="exclude", reason="Duplicate of the fact-owned enrolment_year."),
    dict(feature="field of education in study_area_a / study_area_b",
         source_table="dwh_curriculum__study_area_a, __study_area_b",
         source_column="broad/narrow/detailed_field_of_education", staging_name="-", role="-",
         prediction_time_valid=True, decision="exclude",
         reason="Both join 1:1 and are the only populated attributes in those dimensions, but they "
                "duplicate the field-of-education concept owned by the course dimension. Exclusion "
                "reviewed and confirmed by the human reviewer 2026-08-06; course remains the single "
                "canonical owner."),
    dict(feature="field of education in unit", source_table="dwh_curriculum__unit",
         source_column="broad/narrow/detailed_field_of_education", staging_name="-", role="-",
         prediction_time_valid=True, decision="exclude",
         reason="Duplicate concept, and the unit key is not unique in the current generation, so the "
                "join would inflate the fact table from 973,770 to 7,508,854 rows."),
    dict(feature="course_offering / unit_offering attributes", source_table="offering dimensions",
         source_column="(all non-key columns)", staging_name="-", role="-", prediction_time_valid=True,
         decision="exclude", reason="No populated non-key attributes in the current generation."),
    dict(feature="organisation attributes", source_table=ORGANISATION_TABLE.split(".")[-1],
         source_column="(all non-key columns)", staging_name="-", role="-", prediction_time_valid=True,
         decision="exclude", reason="No populated non-key attributes in the current generation."),
    dict(feature="module / module_offering / thesis attributes", source_table="curriculum dimensions",
         source_column="(all)", staging_name="-", role="-", prediction_time_valid=True,
         decision="exclude",
         reason="curriculum_item_key_hash maps to unit, module and thesis in the relationship model; "
                "the fact keys resolve only to unit, so module and thesis are profiled but not joined."),
    dict(feature="all remaining fact columns", source_table="fact", source_column="(several)",
         staging_name="-", role="-", prediction_time_valid=None, decision="exclude",
         reason="Completely NULL in the current synthetic generation; verified in section 3.2."),
]

manifest_df = spark.createDataFrame(
    [
        (
            m["feature"], m["source_table"], m["source_column"], m["staging_name"], m["role"],
            m["prediction_time_valid"], m["decision"], m["reason"],
        )
        for m in FEATURE_MANIFEST
    ],
    schema=[
        "final_feature_name", "source_table", "source_column", "staging_name", "role",
        "prediction_time_valid", "decision", "reason",
    ],
)
display(manifest_df)

INCLUDED_MANIFEST = [m for m in FEATURE_MANIFEST if m["decision"] == "include"]
NUMERIC_FEATURES = [m["feature"] for m in INCLUDED_MANIFEST if m["role"] == "numeric"]
CATEGORICAL_FEATURES = [m["feature"] for m in INCLUDED_MANIFEST if m["role"] == "categorical"]
FEATURE_COLUMNS = [m["feature"] for m in INCLUDED_MANIFEST]

print(f"Included features: {len(FEATURE_COLUMNS)} ({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical)")
print(f"  numeric    : {NUMERIC_FEATURES}")
print(f"  categorical: {CATEGORICAL_FEATURES}")

## 8. Fact-centred assembly (T2, T3)

The fact projection selects only identifiers, the target, and the fact-owned approved
attributes under source-qualified staging names. Wildcard enrichment is never used.

In [0]:
FACT_STAGING_MAP = {
    m["source_column"]: m["staging_name"]
    for m in INCLUDED_MANIFEST
    if m["source_table"] == "fact"
}

missing_fact_columns = [c for c in FACT_STAGING_MAP if c not in fact_df.columns]
assert not missing_fact_columns, (
    f"Phase 1 stop: approved fact columns not present in the source table -> {missing_fact_columns}"
)

# Join keys are carried through the assembly and dropped before the final projection.
JOIN_KEY_COLUMNS = ["course_key_hash", "teaching_period_key_hash"]

fact_centred_df = fact_df.select(
    F.col(STUDENT_ID_COLUMN),
    F.col(ENROLMENT_ID_COLUMN),
    F.col(TARGET_COLUMN),
    *[F.col(src).alias(staging) for src, staging in FACT_STAGING_MAP.items()],
    *[F.col(k) for k in JOIN_KEY_COLUMNS],
)

print(f"Fact-centred projection columns ({len(fact_centred_df.columns)}):")
for column in fact_centred_df.columns:
    print(f"  {column}")
print(f"Fact-centred row count: {fact_centred_df.count():,}")

### 8.1 Accepted dimension joins

Only two dimensions contribute approved, populated, non-duplicated attributes:
the **course** dimension and the **teaching-period** dimension.

In [0]:
COURSE_ATTRIBUTE_MAP = {
    m["source_column"]: m["staging_name"]
    for m in INCLUDED_MANIFEST
    if m["source_table"] == "dwh_curriculum__course"
}

assembled_df = contracted_left_join(
    base_df=fact_centred_df,
    base_key="course_key_hash",
    dim_fqn=COURSE_TABLE,
    dim_key="course_key_hash",
    attribute_map=COURSE_ATTRIBUTE_MAP,
    alias="course",
)

In [0]:
TEACHING_PERIOD_ATTRIBUTE_MAP = {
    m["source_column"]: m["staging_name"]
    for m in INCLUDED_MANIFEST
    if m["source_table"] == "dwh_learning_and_teaching__teaching_period"
}

assembled_df = contracted_left_join(
    base_df=assembled_df,
    base_key="teaching_period_key_hash",
    dim_fqn=TEACHING_PERIOD_TABLE,
    dim_key="teaching_period_key_hash",
    attribute_map=TEACHING_PERIOD_ATTRIBUTE_MAP,
    alias="teaching_period",
)

### 8.2 Join ledger

In [0]:
join_ledger_df = spark.createDataFrame(
    [
        (
            j["alias"], j["dimension_table"].split(".")[-1], j["fact_key"], j["dimension_key"],
            ", ".join(j["attributes_added"]), j["rows_before"], j["rows_after"],
            j["students_before"], j["students_after"], j["matched_rows"], j["unmatched_rows"],
        )
        for j in JOIN_LEDGER
    ],
    schema=[
        "join", "dimension_table", "fact_key", "dimension_key", "attributes_added",
        "rows_before", "rows_after", "students_before", "students_after",
        "matched_rows", "unmatched_rows",
    ],
)
display(join_ledger_df)

## 9. Final Phase 1 canonical projection

An explicit projection containing the student identifier, the enrolment identifier for
traceability, the target source column, and only the approved canonical predictive columns.
Staging names are resolved to final feature names here.

In [0]:
STAGING_TO_FINAL = {m["staging_name"]: m["feature"] for m in INCLUDED_MANIFEST}

phase1_modelling_df = assembled_df.select(
    F.col(STUDENT_ID_COLUMN),
    F.col(ENROLMENT_ID_COLUMN),
    F.col(TARGET_COLUMN),
    *[F.col(staging).alias(final) for staging, final in STAGING_TO_FINAL.items()],
)

print("Final Phase 1 modelling schema:")
phase1_modelling_df.printSchema()
display(phase1_modelling_df.limit(20))

## 10. Final Phase 1 validation (T4, T5, T6, T8)

In [0]:
# ---- 10.1 One row per student -------------------------------------------------------------------
modelling_stats = phase1_modelling_df.agg(
    F.count(F.lit(1)).alias("rows"),
    F.countDistinct(F.col(STUDENT_ID_COLUMN)).alias("students"),
).collect()[0]
modelling_rows = _as_long(modelling_stats["rows"])
modelling_students = _as_long(modelling_stats["students"])
assert modelling_rows == modelling_students, (
    f"Phase 1 stop: modelling dataset is not one row per student "
    f"({modelling_rows:,} rows vs {modelling_students:,} distinct students)."
)

# ---- 10.2 Unique DataFrame column names ---------------------------------------------------------
duplicate_columns = sorted({c for c in phase1_modelling_df.columns if phase1_modelling_df.columns.count(c) > 1})
assert not duplicate_columns, f"Phase 1 stop: duplicate column names -> {duplicate_columns}"

# ---- 10.3 Unique final feature names ------------------------------------------------------------
assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS)), (
    f"Phase 1 stop: duplicate final feature names -> "
    f"{sorted({f for f in FEATURE_COLUMNS if FEATURE_COLUMNS.count(f) > 1})}"
)

# ---- 10.4 No target, leakage, identifier, raw key, or hash in the feature list -------------------
forbidden_in_features = [
    f for f in FEATURE_COLUMNS
    if f in PROHIBITED_LEAKAGE_COLUMNS
    or f in IDENTIFIER_COLUMNS
    or f.endswith("_key")
    or f.endswith("_key_hash")
    or f.endswith("_hash")
]
assert not forbidden_in_features, (
    f"Phase 1 stop: prohibited columns present in the feature list -> {forbidden_in_features}"
)

# ---- 10.5 Binary, non-null target on supervised rows --------------------------------------------
supervised_df = phase1_modelling_df.filter(F.col(TARGET_COLUMN).isNotNull())
supervised_rows = supervised_df.count()
unsupervised_rows = modelling_rows - supervised_rows

assert supervised_rows > 0, "Phase 1 stop: no supervised rows with a non-null target."

distinct_target_values = {
    row[TARGET_COLUMN] for row in supervised_df.select(TARGET_COLUMN).distinct().collect()
}
assert distinct_target_values.issubset({True, False}), (
    f"Phase 1 stop: non-binary target values -> {distinct_target_values - {True, False}}"
)

print("Phase 1 dataset validation passed.")
print(f"  modelling rows        : {modelling_rows:,}")
print(f"  distinct students     : {modelling_students:,}")
print(f"  supervised rows       : {supervised_rows:,}")
print(f"  rows with NULL target : {unsupervised_rows:,}")
print(f"  feature columns       : {len(FEATURE_COLUMNS)}")

### 10.6 Target class distribution

In [0]:
class_distribution_df = (
    supervised_df.groupBy(F.col(TARGET_COLUMN).alias("class"))
    .count()
    .withColumn("percentage", F.round(100.0 * F.col("count") / F.lit(supervised_rows), 4))
    .orderBy("class")
)
display(class_distribution_df)

positive_rows = supervised_df.filter(F.col(TARGET_COLUMN) == True).count()  # noqa: E712
negative_rows = supervised_rows - positive_rows
print(f"positive class (attrition): {positive_rows:,} ({100.0 * positive_rows / supervised_rows:.2f}%)")
print(f"negative class            : {negative_rows:,} ({100.0 * negative_rows / supervised_rows:.2f}%)")
print(
    "Class imbalance ratio (negative:positive): "
    f"{(negative_rows / positive_rows) if positive_rows else float('nan'):.2f}"
)

### 10.7 Feature missingness and cardinality

Any feature that is completely NULL is removed here. Removing an unusable all-NULL feature is a
permitted validation correction under the plan and is reported in the Phase 1 summary.

In [0]:
feature_counts = phase1_modelling_df.select(*FEATURE_COLUMNS).agg(
    *[F.count(F.col(f"`{c}`")).alias(c) for c in FEATURE_COLUMNS]
).collect()[0].asDict()

cardinalities = {}
if CATEGORICAL_FEATURES:
    cardinality_row = phase1_modelling_df.agg(
        *[F.countDistinct(F.col(f"`{c}`")).alias(c) for c in CATEGORICAL_FEATURES]
    ).collect()[0].asDict()
    cardinalities.update(cardinality_row)

missingness_rows = [
    (
        feature,
        "numeric" if feature in NUMERIC_FEATURES else "categorical",
        int(feature_counts.get(feature, 0) or 0),
        int(modelling_rows - (feature_counts.get(feature, 0) or 0)),
        round(100.0 * (modelling_rows - (feature_counts.get(feature, 0) or 0)) / modelling_rows, 4),
        int(cardinalities.get(feature)) if feature in cardinalities else None,
    )
    for feature in FEATURE_COLUMNS
]

missingness_df = spark.createDataFrame(
    missingness_rows,
    schema=StructType([
        StructField("feature", StringType(), False),
        StructField("role", StringType(), False),
        StructField("non_null_rows", LongType(), True),
        StructField("null_rows", LongType(), True),
        StructField("null_percentage", DoubleType(), True),
        StructField("distinct_values", LongType(), True),
    ]),
)
display(missingness_df.orderBy(F.col("null_percentage").desc()))

ALL_NULL_FEATURES = [f for f in FEATURE_COLUMNS if not feature_counts.get(f)]
if ALL_NULL_FEATURES:
    print(f"Removing completely NULL features (reported as a validation correction): {ALL_NULL_FEATURES}")
    FEATURE_COLUMNS = [f for f in FEATURE_COLUMNS if f not in ALL_NULL_FEATURES]
    NUMERIC_FEATURES = [f for f in NUMERIC_FEATURES if f not in ALL_NULL_FEATURES]
    CATEGORICAL_FEATURES = [f for f in CATEGORICAL_FEATURES if f not in ALL_NULL_FEATURES]
    phase1_modelling_df = phase1_modelling_df.select(
        STUDENT_ID_COLUMN, ENROLMENT_ID_COLUMN, TARGET_COLUMN, *FEATURE_COLUMNS
    )
else:
    print("No completely NULL features in the approved feature set.")

# Constant features carry no signal and are reported (not removed) for reviewer attention.
CONSTANT_FEATURES = [f for f, n in cardinalities.items() if n is not None and n <= 1]
print(f"Single-value categorical features (reported for review): {CONSTANT_FEATURES or 'none'}")

### 10.8 Final exclusion assertion

Confirms one final time that the assembled dataset contains no prohibited column outside the
retained identifiers and the target.

In [0]:
present_columns = set(phase1_modelling_df.columns)
unexpected_leakage = sorted(
    (present_columns & set(PROHIBITED_LEAKAGE_COLUMNS)) - {TARGET_COLUMN}
)
assert not unexpected_leakage, f"Phase 1 stop: leakage columns present -> {unexpected_leakage}"

unexpected_keys = sorted(
    c for c in present_columns
    if (c.endswith("_key") or c.endswith("_key_hash") or c.endswith("_hash"))
    and c not in IDENTIFIER_COLUMNS
)
assert not unexpected_keys, f"Phase 1 stop: raw keys or hashes present -> {unexpected_keys}"

print("Exclusion assertion passed.")
print(f"Final Phase 1 columns ({len(phase1_modelling_df.columns)}): {phase1_modelling_df.columns}")

## 11. Phase 1 result summary

Prints the conclusions the human reviewer needs. Phase 1 produces no persisted output; the
assembled DataFrame `phase1_modelling_df` is the hand-off object for Phase 2.

In [0]:
print("=" * 96)
print("PHASE 1 RESULT - FACT-CENTRED STAR-SCHEMA FEATURE ASSEMBLY")
print("=" * 96)
print(f"Run timestamp (UTC)          : {PHASE_1_RUN_TIMESTAMP}")
print(f"Source tables profiled       : {len(SOURCE_PROFILES)} of 12")
print(f"Fact table                   : {FACT_TABLE}")
print(f"Fact rows                    : {fact_row_count:,}")
print(f"Distinct students            : {distinct_students:,}")
print(f"Modelling grain              : one row per {STUDENT_ID_COLUMN} (verified)")
print(f"Dimensions joined            : course, teaching_period")
print(f"Dimensions excluded          : course_offering, unit, unit_offering, study_area_a, "
      f"study_area_b, organisation, module, module_offering, thesis")
print(f"Target                       : {TARGET_COLUMN}")
print(f"Supervised rows              : {supervised_rows:,}")
print(f"Positive class rate          : {100.0 * positive_rows / supervised_rows:.2f}%")
print(f"Final features               : {len(FEATURE_COLUMNS)} "
      f"({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical)")
print(f"  numeric                    : {NUMERIC_FEATURES}")
print(f"  categorical                : {CATEGORICAL_FEATURES}")
print(f"All-NULL features removed    : {ALL_NULL_FEATURES or 'none'}")
print(f"Single-value features        : {CONSTANT_FEATURES or 'none'}")
for j in JOIN_LEDGER:
    print(
        f"Join {j['alias']:<16}: rows {j['rows_before']:,} -> {j['rows_after']:,}, "
        f"students {j['students_before']:,} -> {j['students_after']:,}, "
        f"unmatched {j['unmatched_rows']:,}"
    )
print("-" * 96)
print("PHASE 1 COMPLETE. Stop here.")
print("Do not create train/validation/test splits or train a model until Phase 1 is approved.")
print("=" * 96)

## 11.1 Executed evidence (run 2026-08-06 against `workspace.student_aggregate`)

The figures below were obtained by running these checks against the live Delta tables. They are
recorded here so the human reviewer can compare them with the output of a fresh notebook run.

**Fact grain**

| Check | Result |
|---|---|
| Fact rows | 973,770 |
| Distinct students | 973,770 |
| Distinct enrolments | 973,770 |
| NULL student / enrolment hashes | 0 / 0 |
| NULL target values | 0 |
| Students with more than one fact row | 0 |

One eligible fact row per student is confirmed, so no eligibility rule or deduplication is needed.

**Target class distribution:** 53,694 positive (5.51%) and 920,076 negative (94.49%), an imbalance
ratio of about 17:1. Phase 2 will therefore need the training weight column described in the plan.

**Accepted joins:** both preserved the grain exactly - 973,770 rows and 973,770 distinct students
before and after, with zero unmatched keys for the course and teaching-period dimensions.

**Feature quality:** all 21 approved features are 100% populated (zero missingness). Categorical
cardinality is modest: socioeconomic status 4, regional/remote 6, gender 5, attendance mode 3,
commencing/continuing 2, commencing/continuing by period 2, course-admission load category 2,
course group 4, broad field of education 10, narrow field of education 39, detailed field of
education 77, teaching period 81. Total one-hot width is roughly 235 columns, which is
comfortable for the approved Random Forest.

### Exhaustive source coverage

The 12 tables were profiled column by column to confirm that no further usable attribute exists.
The synthetic-data generator writes a typed NULL for every column it cannot ground in a supplied
distribution, and that is most of the schema:

- 38 further fact columns tested (`status`, `stage`, `study_mode`, `load_category`,
  `student_fee_status`, `liability_category`, `funding_source`, `is_on_plan`, all `citizenship_*`,
  `student_birth_country`, `student_home_language`, all `student_highest_level_of_achievement*`,
  all `student_first_known_*`, every other `course_admission_*` field, all `*_on_plan` credit
  points, `is_tcsi_*`, `is_withdrawn_*`, `curriculum_item_*`) return **zero** non-null rows.
- Across every curriculum dimension the only populated non-key columns are the three
  field-of-education levels. `unit_type`, `study_area_a_type`, `study_area_b_type`,
  `module_type`, `thesis_type`, `status`, and `total_credit_value` are all entirely NULL.
- The organisation hierarchy has no populated attributes at all - only its key columns.
- `module`, `module_offering`, and `thesis` match zero fact rows, so they cannot contribute.

The 21 assembled features therefore represent complete coverage of the usable STAR schema.

**Leakage confirmation:** every row with `is_twelve_month_course_attrition = false` also has
`is_twelve_month_student_attrition = false` (747,744 rows). Course attrition alone would identify
76.8% of the population as guaranteed negatives, which confirms the specification's requirement to
exclude it from the feature vector.

### Known generator defect - duplicate dimension keys

Three dimensions do **not** have unique business keys, and joining them would multiply fact rows:

| Dimension | Rows | Distinct keys | Fact rows if joined |
|---|---|---|---|
| `dwh_curriculum__course_offering` | 30,516 | 9,999 | 8,167,683 |
| `dwh_curriculum__unit` | 25,733 | 9,999 | 7,508,854 |
| `dwh_curriculum__unit_offering` | 161,916 | 9,999 | (same defect) |

The synthetic-data generator builds keys with `lpad(index, 4, "0")`. Spark's `lpad` truncates an
input longer than the requested length, so every index above 9,999 collapses onto an existing
four-character value. Any dimension with more than 9,999 rows therefore has colliding keys.

Joining any of those three dimensions would multiply fact rows and destroy the one-row-per-student
grain. The join contract in section 6 asserts key uniqueness and would stop Phase 1 if attempted.
The dimensions actually joined - course (1,544 rows) and teaching period (3,083 rows) - are both
below the threshold and have fully unique keys, so the assembled dataset is unaffected.

The defect belongs to the data-generation notebook, not to this workflow. The fix has been applied
there (`KEY_WIDTH` raised from 4 to 12, plus a width guard and a post-write key-uniqueness
assertion), and the reviewer elected to regenerate the Delta tables immediately.

**The figures in this section were measured before regeneration.** They are expected to be
unchanged afterwards, because the deidentified student and enrolment hashes derive from the row id
rather than the padded keys, and dbldatagen seeds its random columns deterministically by default,
so only the synthetic key strings change. After regeneration the three affected dimensions will
have unique keys and become joinable 1:1, but they remain excluded: they contribute only
field-of-education columns that the course dimension already owns. Re-run this notebook against
the regenerated tables to confirm the figures before starting Phase 2.

**Eligibility flags unavailable:** `is_latest_course_enrolment_with_study_load` and
`is_major_course_for_census_date` are entirely NULL, so neither could have been used to resolve a
multi-row grain had one existed.

### Approved refinements (human review, 2026-08-06)

| Refinement | Decision |
|---|---|
| Add `course_admission_load_category` to the feature set | **Approved** - material change, `plan.md` updated |
| Add `commencing_continuing_period` to the feature set | **Approved** - material change, `plan.md` updated |
| Include Study Area A/B field of education | **Rejected** - course remains the single canonical FoE owner |
| Include redundant and zero-variance columns | **Rejected** - no information gain |
| Fix the generator key collision | **Approved** - fix applied and tables regenerated |

The feature set moves from 19 to 21 as a result.

## 12. Phase 1 human review outcome - approved 2026-08-06

The human reviewer approved Phase 1 on the evidence of a complete top-to-bottom run of this
notebook against the regenerated Delta tables, with every validation assertion passing. Tasks
**T1-T8** are marked complete in `tasks.md`, and the re-run requirement recorded in amendment A-3
is discharged.

**Routine correction reported at this gate.** `cache()` was removed from four places. Serverless
compute rejects it with `NOT_SUPPORTED_WITH_SERVERLESS`. The affected counts were folded into
single aggregates so each check costs one pass instead of one pass per figure. No approved
behaviour changed.

**Known limitations recorded (T31).** Three properties of the synthetic data constrain what any
model trained on it can achieve. They are recorded in full as L-1, L-2, and L-3 in `plan.md`, and
summarised here because they govern how the Phase 2 metrics below must be read:

1. **Field of education is decoupled from the target.** The attrition Beta parameters are chosen
   by each fact row's own `broad_field_of_education`, which is never written to the fact table.
   The course dimension's field of education is seeded from a different fact row and coincides
   with the true value on only about 0.16% of rows.
2. **All 21 features are close to flat against the target.** Against a 5.51% base rate the widest
   spread on any feature is 1.16 percentage points, most of which is sampling noise. Since
   E[p] = 23.211% (course attrition) and E[p²] = 5.514% (student attrition), SD(p) is 0.0356 and
   each student's true probability is p² with mean 0.055. **Expect ROC-AUC near 0.5 and precision
   near the base rate.** That is the correct result for this data, not a defect, and it is not a
   reason to alter the approved features, parameters, or threshold.
3. **Weighted probabilities are not calibrated.** Class weighting recentres probabilities on 0.50
   so the approved threshold sits at the decision boundary, but the output is then a relative
   ranking score rather than a likelihood of attrition.

# Phase 2: Split, preprocessing, Random Forest training, MLflow, evaluation

Implements tasks **T9-T21** against the approved Phase 1 canonical modelling projection. No
feature is rediscovered or added here: `phase1_modelling_df`, `FEATURE_COLUMNS`,
`NUMERIC_FEATURES`, and `CATEGORICAL_FEATURES` are consumed exactly as Phase 1 produced them.

Phase 2 trains and evaluates only. It writes no Delta table.

## 13. Phase 2 configuration and approved-input contract (T9, T15)

Every Phase 2 constant is declared once here. The contract assertions stop the notebook if the
approved Phase 1 columns are missing or have changed, rather than silently substituting features.

In [0]:
import gc
import os
import sys
import uuid

import mlflow
import mlflow.spark

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import vector_to_array

# ---- Split. Fractions and seed are the Phase 1 constants; nothing is redeclared here. -----------
SPLIT_SEED = RANDOM_SEED
SPLIT_BUCKETS = 100
TRAIN_BUCKET_LIMIT = int(TRAIN_FRACTION * SPLIT_BUCKETS)                               # buckets  0-69
VALIDATION_BUCKET_LIMIT = int((TRAIN_FRACTION + VALIDATION_FRACTION) * SPLIT_BUCKETS)  # buckets 70-84

# ---- Random Forest, exactly as approved in plan.md section 8 ------------------------------------
RF_NUM_TREES = 100
RF_MAX_DEPTH = 7
RF_MIN_INSTANCES_PER_NODE = 20
RF_FEATURE_SUBSET_STRATEGY = "sqrt"
RF_SEED = RANDOM_SEED

# ---- Working column names. Prefixed to guarantee no collision with a feature name. --------------
LABEL_COLUMN = "__label"
WEIGHT_COLUMN = "__class_weight"
SPLIT_COLUMN = "__split"
FEATURES_VECTOR_COLUMN = "__features"
PROBABILITY_COLUMN = "__probability"
RAW_PREDICTION_COLUMN = "__raw_prediction"
PREDICTION_COLUMN = "__prediction"
POSITIVE_PROBABILITY_COLUMN = "__positive_probability"

MLFLOW_RUN_NAME = "sprint2_student_attrition_random_forest"

# ---- Approved-input contract (Phase 2 prompt: stop rather than substitute) -----------------------
_missing_phase1 = [
    name for name in ("phase1_modelling_df", "FEATURE_COLUMNS", "NUMERIC_FEATURES",
                      "CATEGORICAL_FEATURES", "supervised_rows", "FEATURE_MANIFEST")
    if name not in globals()
]
assert not _missing_phase1, (
    f"Phase 2 stop: Phase 1 outputs not in scope -> {_missing_phase1}. Run this notebook from the top."
)

_available_columns = set(phase1_modelling_df.columns)
_absent_features = [f for f in FEATURE_COLUMNS if f not in _available_columns]
assert not _absent_features, (
    f"Phase 2 stop: approved Phase 1 features absent from the modelling projection -> {_absent_features}. "
    "Report the conflict; do not substitute features."
)
assert TARGET_COLUMN in _available_columns, f"Phase 2 stop: target {TARGET_COLUMN} absent."
assert STUDENT_ID_COLUMN in _available_columns, f"Phase 2 stop: {STUDENT_ID_COLUMN} absent."
assert sorted(FEATURE_COLUMNS) == sorted(NUMERIC_FEATURES + CATEGORICAL_FEATURES), (
    "Phase 2 stop: the numeric and categorical lists do not reconstruct the approved feature list."
)
assert abs((TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION) - 1.0) < 1e-9, (
    "Phase 2 stop: split fractions do not sum to 1."
)

print(f"Approved Phase 1 manifest accepted: {len(FEATURE_COLUMNS)} features "
      f"({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical).")
print(f"Target: {TARGET_COLUMN}")
print(f"Threshold (applied in evaluation and Phase 3): {RISK_THRESHOLD}")

## 14. Deterministic 70/15/15 split (T9)

The split is assigned by hashing `student_deidentified_hash` together with seed 42 into 100
buckets: 0-69 train, 70-84 validation, 85-99 test.

**Why not `randomSplit`.** Spark's `randomSplit` samples per partition, so its result is stable
only while the physical plan and partitioning are stable; the documented mitigation is to cache or
save the DataFrame before splitting. Serverless compute forbids `cache()`, and this DataFrame is
recomputed on every action, so a plan change between the fit action and an evaluation action could
silently move rows between splits and leak test data into training. Hashing the student identifier
is a pure function of the data: it gives the same assignment on every action, on every cluster, and
after any replan, and it makes a student appearing in two splits arithmetically impossible.

This changes *which* rows land in each split relative to a literal `randomSplit(seed=42)`, so it is
raised as a material refinement for approval in the Phase 2 summary. Ratios, seed, determinism, and
disjointness all remain as approved.

In [0]:
SPLIT_BUCKET_COLUMN = "__split_bucket"

split_assigned_df = (
    phase1_modelling_df.withColumn(
        SPLIT_BUCKET_COLUMN,
        F.pmod(
            F.hash(F.concat_ws("|", F.col(STUDENT_ID_COLUMN), F.lit(SPLIT_SEED))),
            F.lit(SPLIT_BUCKETS),
        ),
    )
    .withColumn(
        SPLIT_COLUMN,
        F.when(F.col(SPLIT_BUCKET_COLUMN) < F.lit(TRAIN_BUCKET_LIMIT), F.lit("train"))
        .when(F.col(SPLIT_BUCKET_COLUMN) < F.lit(VALIDATION_BUCKET_LIMIT), F.lit("validation"))
        .otherwise(F.lit("test")),
    )
    .withColumn(LABEL_COLUMN, F.col(TARGET_COLUMN).cast("double"))
)

# Supervised rows only: a row with no label cannot train or evaluate. Phase 1 confirmed the target is
# fully populated, so this is a guard rather than a filter that is expected to remove anything.
supervised_split_df = split_assigned_df.filter(F.col(LABEL_COLUMN).isNotNull())

train_df = supervised_split_df.filter(F.col(SPLIT_COLUMN) == F.lit("train"))
validation_df = supervised_split_df.filter(F.col(SPLIT_COLUMN) == F.lit("validation"))
test_df = supervised_split_df.filter(F.col(SPLIT_COLUMN) == F.lit("test"))

print("Split assigned by hash bucket: train 0-69, validation 70-84, test 85-99.")

### 14.1 Split row counts and class distributions (T10)

In [0]:
# One aggregate over the whole dataset produces every split's counts at once (no cache on serverless).
split_stats_rows = (
    supervised_split_df.groupBy(SPLIT_COLUMN)
    .agg(
        F.count(F.lit(1)).alias("rows"),
        F.countDistinct(F.col(STUDENT_ID_COLUMN)).alias("distinct_students"),
        F.count(F.when(F.col(LABEL_COLUMN) == F.lit(1.0), F.lit(1))).alias("positive_rows"),
        F.count(F.when(F.col(LABEL_COLUMN) == F.lit(0.0), F.lit(1))).alias("negative_rows"),
    )
    .collect()
)

SPLIT_STATS = {row[SPLIT_COLUMN]: row.asDict() for row in split_stats_rows}

_empty_splits = [name for name in ("train", "validation", "test") if name not in SPLIT_STATS]
assert not _empty_splits, (
    f"Phase 2 stop: no rows were assigned to {_empty_splits}. The split cannot proceed."
)

total_supervised_rows = sum(_as_long(s["rows"]) for s in SPLIT_STATS.values())

split_summary_df = spark.createDataFrame(
    [
        (
            name,
            _as_long(SPLIT_STATS[name]["rows"]),
            round(100.0 * SPLIT_STATS[name]["rows"] / total_supervised_rows, 4),
            _as_long(SPLIT_STATS[name]["positive_rows"]),
            _as_long(SPLIT_STATS[name]["negative_rows"]),
            round(100.0 * SPLIT_STATS[name]["positive_rows"] / SPLIT_STATS[name]["rows"], 4),
        )
        for name in ("train", "validation", "test")
    ],
    schema=StructType([
        StructField("split", StringType(), False),
        StructField("rows", LongType(), True),
        StructField("share_of_total_pct", DoubleType(), True),
        StructField("positive_rows", LongType(), True),
        StructField("negative_rows", LongType(), True),
        StructField("positive_class_pct", DoubleType(), True),
    ]),
)
display(split_summary_df)

### 14.2 Split integrity assertions (T10)

Every split must be non-empty, carry both classes, and share no student with another split.

In [0]:
for _split_name in ("train", "validation", "test"):
    assert _split_name in SPLIT_STATS, f"Phase 2 stop: split '{_split_name}' is empty."
    _stats = SPLIT_STATS[_split_name]
    assert _stats["rows"] == _stats["distinct_students"], (
        f"Phase 2 stop: split '{_split_name}' is not one row per student "
        f"({_stats['rows']} rows vs {_stats['distinct_students']} students)."
    )
    assert _stats["positive_rows"] > 0 and _stats["negative_rows"] > 0, (
        f"Phase 2 stop: split '{_split_name}' does not contain both classes."
    )

# A student assigned to two splits is impossible by construction, but the claim is worth proving
# rather than asserting: count students whose rows span more than one split label.
students_in_multiple_splits = (
    supervised_split_df.groupBy(STUDENT_ID_COLUMN)
    .agg(F.countDistinct(F.col(SPLIT_COLUMN)).alias("split_count"))
    .filter(F.col("split_count") > 1)
    .count()
)
assert students_in_multiple_splits == 0, (
    f"Phase 2 stop: {students_in_multiple_splits} students appear in more than one split."
)

assert total_supervised_rows == supervised_rows, (
    f"Phase 2 stop: split total {total_supervised_rows:,} does not match the supervised row count "
    f"established in Phase 1 ({supervised_rows:,}). Rows were lost or duplicated during split assignment."
)

_train_share = 100.0 * SPLIT_STATS["train"]["rows"] / total_supervised_rows
assert 68.0 <= _train_share <= 72.0, (
    f"Phase 2 stop: training share {_train_share:.2f}% is not approximately 70%."
)

print(
    f"Split integrity confirmed: {total_supervised_rows:,} supervised rows, no student in more than "
    "one split, both classes present in all three splits."
)

## 15. Preprocessing (T11, T12, T13)

Preprocessing metadata is learned once from the training split only, in section 15 below, then
applied to every split with plain Spark column expressions. No Spark ML Pipeline is created.

- **Numeric:** cast to double; any NULL or NaN value is replaced with the training-split median.
- **Categorical:** cast to string, then represented as one binary column per category value seen
  in the training split, plus one reserved column for NULL or previously unseen values.
- **Assembly:** the derived columns are combined into the feature vector by `VectorAssembler`,
  used only as a Transformer.
- **No scaling.** The approved Random Forest does not require it.

In [0]:
# ======================================================================================
# Stateless preprocessing
#
# Learn small Python preprocessing metadata (numeric medians, categorical levels) from the
# TRAINING split only, then apply it using ordinary Spark column expressions.
# ======================================================================================

NUMERIC_VECTOR_SUFFIX = "__numeric"
UNKNOWN_CATEGORY_SUFFIX = "__unknown"


# --------------------------------------------------------------------------------------
# 1. Training-only numeric medians
# --------------------------------------------------------------------------------------
# Median values are computed once from the training split and reused for every split.

numeric_median_row = train_df.agg(
    *[
        F.percentile_approx(
            F.col(feature).cast("double"),
            0.5,
            10000
        ).alias(feature)
        for feature in NUMERIC_FEATURES
    ]
).collect()[0].asDict()

NUMERIC_MEDIANS = {
    feature: float(numeric_median_row[feature])
    for feature in NUMERIC_FEATURES
}

assert all(value is not None for value in NUMERIC_MEDIANS.values()), (
    "Phase 2 stop: at least one numeric feature has no usable training median."
)


# --------------------------------------------------------------------------------------
# 2. Training-only categorical domains
# --------------------------------------------------------------------------------------
# Category values are ranked by frequency (descending), with an alphabetical tie-break to keep
# the ordering deterministic.

CATEGORY_LEVELS = {}

for feature in CATEGORICAL_FEATURES:
    category_rows = (
        train_df
        .select(F.col(feature).cast("string").alias("__category"))
        .where(F.col("__category").isNotNull())
        .groupBy("__category")
        .count()
        .orderBy(
            F.col("count").desc(),
            F.col("__category").asc()
        )
        .collect()
    )

    CATEGORY_LEVELS[feature] = [
        row["__category"]
        for row in category_rows
    ]

    assert CATEGORY_LEVELS[feature], (
        f"Phase 2 stop: categorical feature {feature} has no training categories."
    )


# --------------------------------------------------------------------------------------
# 3. Define the exact vector columns emitted for every approved feature
# --------------------------------------------------------------------------------------

FEATURE_VECTOR_COLUMNS_BY_FEATURE = {}

for feature in NUMERIC_FEATURES:
    FEATURE_VECTOR_COLUMNS_BY_FEATURE[feature] = [
        f"{feature}{NUMERIC_VECTOR_SUFFIX}"
    ]

for feature in CATEGORICAL_FEATURES:
    known_columns = [
        f"{feature}__category_{index:03d}"
        for index in range(len(CATEGORY_LEVELS[feature]))
    ]

    FEATURE_VECTOR_COLUMNS_BY_FEATURE[feature] = (
        known_columns
        + [f"{feature}{UNKNOWN_CATEGORY_SUFFIX}"]
    )


# Preserve a deterministic assembler ordering.
ASSEMBLER_INPUT_COLUMNS = []

for feature in NUMERIC_FEATURES + CATEGORICAL_FEATURES:
    ASSEMBLER_INPUT_COLUMNS.extend(
        FEATURE_VECTOR_COLUMNS_BY_FEATURE[feature]
    )


# --------------------------------------------------------------------------------------
# 4. Stateless Spark transformation
# --------------------------------------------------------------------------------------

def add_preprocessing_columns(df):
    """
    Add the numeric-imputed and categorical one-hot columns using Spark SQL expressions.
    """

    derived_columns = []

    # Numeric features
    for feature in NUMERIC_FEATURES:
        numeric_value = F.col(feature).cast("double")
        median_value = NUMERIC_MEDIANS[feature]

        derived_columns.append(
            F.when(
                numeric_value.isNull() | F.isnan(numeric_value),
                F.lit(median_value)
            )
            .otherwise(numeric_value)
            .alias(f"{feature}{NUMERIC_VECTOR_SUFFIX}")
        )

    # Categorical features
    for feature in CATEGORICAL_FEATURES:
        string_value = F.col(feature).cast("string")
        known_values = CATEGORY_LEVELS[feature]

        # One binary column for every category learned from training.
        for index, category_value in enumerate(known_values):
            derived_columns.append(
                F.when(
                    string_value == F.lit(category_value),
                    F.lit(1.0)
                )
                .otherwise(F.lit(0.0))
                .alias(f"{feature}__category_{index:03d}")
            )

        # One explicit bucket for NULL or previously unseen values.
        if known_values:
            unknown_condition = (
                string_value.isNull()
                | (~string_value.isin(*known_values))
            )
        else:
            unknown_condition = F.lit(True)

        derived_columns.append(
            F.when(unknown_condition, F.lit(1.0))
            .otherwise(F.lit(0.0))
            .alias(f"{feature}{UNKNOWN_CATEGORY_SUFFIX}")
        )

    return df.select("*", *derived_columns)


# --------------------------------------------------------------------------------------
# 5. VectorAssembler is a Transformer only -- it does not call fit()
# --------------------------------------------------------------------------------------

vector_assembler = VectorAssembler(
    inputCols=ASSEMBLER_INPUT_COLUMNS,
    outputCol=FEATURES_VECTOR_COLUMN,
    handleInvalid="error",
)


def prepare_feature_vector(df):
    """Apply stateless preprocessing and assemble the Random Forest feature vector."""
    return vector_assembler.transform(
        add_preprocessing_columns(df)
    )


print("Stateless preprocessing configured.")
print(f"Numeric features          : {len(NUMERIC_FEATURES)}")
print(f"Categorical features      : {len(CATEGORICAL_FEATURES)}")
print(f"Final vector width        : {len(ASSEMBLER_INPUT_COLUMNS)}")

for feature in CATEGORICAL_FEATURES:
    print(
        f"  {feature:<45}: "
        f"{len(CATEGORY_LEVELS[feature])} known categories + 1 unknown bucket"
    )

### 15.1 Feature-vector exclusion assertion (T13)

Proves that no identifier, label, leakage field, raw key, or hash can reach the features vector.
The check runs directly against the approved feature names, since the stateless preprocessing
metadata already keys every generated column back to the feature it came from.

In [0]:
assert set(FEATURE_VECTOR_COLUMNS_BY_FEATURE) == set(FEATURE_COLUMNS), (
    "Phase 2 stop: stateless preprocessing does not cover exactly the approved features."
)

forbidden_features = sorted(
    feature
    for feature in FEATURE_COLUMNS
    if feature in PROHIBITED_LEAKAGE_COLUMNS
    or feature in IDENTIFIER_COLUMNS
    or feature == TARGET_COLUMN
    or feature in (LABEL_COLUMN, WEIGHT_COLUMN, SPLIT_COLUMN)
    or feature.endswith("_key")
    or feature.endswith("_key_hash")
    or feature.endswith("_hash")
)

assert not forbidden_features, (
    f"Phase 2 stop: prohibited feature reached preprocessing -> {forbidden_features}"
)

assert len(ASSEMBLER_INPUT_COLUMNS) == len(set(ASSEMBLER_INPUT_COLUMNS)), (
    "Phase 2 stop: duplicate derived feature-vector column names."
)

print(
    f"Feature vector confirmed: {len(FEATURE_COLUMNS)} approved source features, "
    f"{len(ASSEMBLER_INPUT_COLUMNS)} assembled numeric slots, "
    "with no identifiers, leakage fields, raw keys, or hashes."
)

## 16. Class imbalance and the training weight column (T14)

The class distribution is taken from the **training split only**, so no validation or test
information influences the weights.

Formula, the standard balanced weighting:

```
weight(class c) = training_rows / (2 x training_rows_in_class_c)
```

Each class then contributes the same total weight, and the weights sum to the training row count.
Weighting is applied only when the positive class is materially under-represented, defined here as
a positive share below 40%.

In [0]:
MATERIAL_IMBALANCE_THRESHOLD_PCT = 40.0

train_rows = _as_long(SPLIT_STATS["train"]["rows"])
train_positive_rows = _as_long(SPLIT_STATS["train"]["positive_rows"])
train_negative_rows = _as_long(SPLIT_STATS["train"]["negative_rows"])
train_positive_pct = 100.0 * train_positive_rows / train_rows

APPLY_CLASS_WEIGHTS = train_positive_pct < MATERIAL_IMBALANCE_THRESHOLD_PCT

if APPLY_CLASS_WEIGHTS:
    POSITIVE_CLASS_WEIGHT = train_rows / (2.0 * train_positive_rows)
    NEGATIVE_CLASS_WEIGHT = train_rows / (2.0 * train_negative_rows)
else:
    POSITIVE_CLASS_WEIGHT = 1.0
    NEGATIVE_CLASS_WEIGHT = 1.0

print(f"Training positive class: {train_positive_rows:,} of {train_rows:,} rows ({train_positive_pct:.4f}%)")
print(f"Materially under-represented (< {MATERIAL_IMBALANCE_THRESHOLD_PCT}%): {APPLY_CLASS_WEIGHTS}")
if APPLY_CLASS_WEIGHTS:
    print(f"  positive weight: {POSITIVE_CLASS_WEIGHT:.6f}")
    print(f"  negative weight: {NEGATIVE_CLASS_WEIGHT:.6f}")
    print(f"  ratio (positive:negative): {POSITIVE_CLASS_WEIGHT / NEGATIVE_CLASS_WEIGHT:.4f}")
    print(
        "  Note: weighting recentres predicted probabilities on 0.50 so the approved threshold sits at "
        "the decision boundary. Output is then a ranking score, not a calibrated probability (plan L-3)."
    )
else:
    print("  Weighting not applied: the positive class is not materially under-represented.")

def with_model_columns(df):
    """Create the final feature vector and attach the class-weight column."""
    return prepare_feature_vector(df).withColumn(
        WEIGHT_COLUMN,
        F.when(
            F.col(LABEL_COLUMN) == F.lit(1.0),
            F.lit(POSITIVE_CLASS_WEIGHT)
        )
        .otherwise(F.lit(NEGATIVE_CLASS_WEIGHT)),
    )


train_prepared_df = with_model_columns(train_df)
validation_prepared_df = with_model_columns(validation_df)
test_prepared_df = with_model_columns(test_df)

## 17. Random Forest configuration (T15)

Exactly the parameters approved in `plan.md` section 8. No model comparison, no competing feature
subsets, no alternative parameter configurations, and no hyperparameter search.

In [0]:
random_forest = RandomForestClassifier(
    featuresCol=FEATURES_VECTOR_COLUMN,
    labelCol=LABEL_COLUMN,
    predictionCol=PREDICTION_COLUMN,
    probabilityCol=PROBABILITY_COLUMN,
    rawPredictionCol=RAW_PREDICTION_COLUMN,
    numTrees=RF_NUM_TREES,
    maxDepth=RF_MAX_DEPTH,
    minInstancesPerNode=RF_MIN_INSTANCES_PER_NODE,
    featureSubsetStrategy=RF_FEATURE_SUBSET_STRATEGY,
    seed=RF_SEED,
)

if APPLY_CLASS_WEIGHTS:
    random_forest = random_forest.setWeightCol(WEIGHT_COLUMN)

RF_PARAMETERS = {
    "numTrees": RF_NUM_TREES,
    "maxDepth": RF_MAX_DEPTH,
    "minInstancesPerNode": RF_MIN_INSTANCES_PER_NODE,
    "featureSubsetStrategy": RF_FEATURE_SUBSET_STRATEGY,
    "seed": RF_SEED,
    "weightCol": WEIGHT_COLUMN if APPLY_CLASS_WEIGHTS else None,
}
for _name, _value in RF_PARAMETERS.items():
    print(f"  {_name}: {_value}")

## 18. Model assembly (T16)

In [0]:
print("=" * 80)
print("MODEL ASSEMBLY")
print("=" * 80)
print("Preprocessing      : stateless Spark column expressions")
print("Vector assembly    : VectorAssembler.transform()")
print("Fitted ML object   : RandomForestClassificationModel only")
print("Spark ML Pipeline  : not used")

## 19. Evaluation helpers (T19, T20, T21)

One function computes every required metric so validation and test are measured identically.
Recall, precision, and F1 are reported **for the positive attrition class**, not weighted
averages, because a weighted average over a 94.5% negative class would be dominated by the
majority and would misrepresent the primary metric.

The predicted class is derived from the approved 0.50 threshold applied to the positive-class
probability, rather than read from the classifier's own `prediction` column, so the reported
metrics are the ones the Phase 3 risk flag will reproduce.

In [0]:
def score_split(model, prepared_df):
    """Apply the fitted Random Forest model and derive the positive probability and thresholded prediction."""
    scored = model.transform(prepared_df)
    return scored.withColumn(
        POSITIVE_PROBABILITY_COLUMN, vector_to_array(F.col(PROBABILITY_COLUMN))[1]
    ).withColumn(
        PREDICTION_COLUMN,
        (F.col(POSITIVE_PROBABILITY_COLUMN) >= F.lit(RISK_THRESHOLD)).cast("double"),
    )


def evaluate_split(scored_df, split_name: str) -> dict:
    """Confusion-matrix counts, the six required metrics and MSE, and probability-spread diagnostics."""
    counts = scored_df.agg(
        F.count(F.lit(1)).alias("rows"),
        F.count(F.when((F.col(LABEL_COLUMN) == 1.0) & (F.col(PREDICTION_COLUMN) == 1.0), F.lit(1))).alias("tp"),
        F.count(F.when((F.col(LABEL_COLUMN) == 0.0) & (F.col(PREDICTION_COLUMN) == 0.0), F.lit(1))).alias("tn"),
        F.count(F.when((F.col(LABEL_COLUMN) == 0.0) & (F.col(PREDICTION_COLUMN) == 1.0), F.lit(1))).alias("fp"),
        F.count(F.when((F.col(LABEL_COLUMN) == 1.0) & (F.col(PREDICTION_COLUMN) == 0.0), F.lit(1))).alias("fn"),
        F.avg(F.pow(F.col(LABEL_COLUMN) - F.col(POSITIVE_PROBABILITY_COLUMN), 2)).alias("mse"),
        F.count(F.when(F.col(PREDICTION_COLUMN) == 1.0, F.lit(1))).alias("predicted_positive"),
        F.count(F.when(F.col(PREDICTION_COLUMN) == 0.0, F.lit(1))).alias("predicted_negative"),
        F.min(F.col(POSITIVE_PROBABILITY_COLUMN)).alias("min_probability"),
        F.max(F.col(POSITIVE_PROBABILITY_COLUMN)).alias("max_probability"),
        F.avg(F.col(POSITIVE_PROBABILITY_COLUMN)).alias("mean_probability"),
        F.stddev(F.col(POSITIVE_PROBABILITY_COLUMN)).alias("stddev_probability"),
        F.countDistinct(F.round(F.col(POSITIVE_PROBABILITY_COLUMN), 6)).alias("distinct_probabilities"),
    ).collect()[0]

    tp, tn = _as_long(counts["tp"]), _as_long(counts["tn"])
    fp, fn = _as_long(counts["fp"]), _as_long(counts["fn"])
    rows = _as_long(counts["rows"])

    recall = tp / (tp + fn) if (tp + fn) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    accuracy = (tp + tn) / rows if rows else 0.0

    # ROC-AUC is threshold-independent, so it is taken from the continuous probability.
    roc_auc = BinaryClassificationEvaluator(
        labelCol=LABEL_COLUMN, rawPredictionCol=POSITIVE_PROBABILITY_COLUMN, metricName="areaUnderROC"
    ).evaluate(scored_df)

    return {
        "split": split_name,
        "rows": rows,
        "recall": recall,
        "accuracy": accuracy,
        "precision": precision,
        "f1": f1,
        "roc_auc": roc_auc,
        "true_positives": tp,
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "mse": float(counts["mse"]),
        "predicted_positive": _as_long(counts["predicted_positive"]),
        "predicted_negative": _as_long(counts["predicted_negative"]),
        "min_probability": float(counts["min_probability"]),
        "max_probability": float(counts["max_probability"]),
        "mean_probability": float(counts["mean_probability"]),
        "stddev_probability": float(counts["stddev_probability"] or 0.0),
        "distinct_probabilities": _as_long(counts["distinct_probabilities"]),
    }


def metrics_to_dataframe(metric_dicts: list):
    return spark.createDataFrame(
        [
            (
                m["split"],
                _as_long(m["rows"]),
                round(m["recall"], 6),
                round(m["accuracy"], 6),
                round(m["precision"], 6),
                round(m["f1"], 6),
                round(m["roc_auc"], 6),
                round(m["mse"], 6),  # ADD THIS
                _as_long(m["true_positives"]),
                _as_long(m["true_negatives"]),
                _as_long(m["false_positives"]),
                _as_long(m["false_negatives"]),
                _as_long(m["predicted_positive"]),
                _as_long(m["predicted_negative"]),
            )
            for m in metric_dicts
        ],
        schema=StructType([
            StructField("split", StringType(), False),
            StructField("rows", LongType(), True),
            StructField("recall_primary", DoubleType(), True),
            StructField("accuracy", DoubleType(), True),
            StructField("precision", DoubleType(), True),
            StructField("f1", DoubleType(), True),
            StructField("roc_auc", DoubleType(), True),
            StructField("mse", DoubleType(), True),  # ADD THIS
            StructField("true_positives", LongType(), True),
            StructField("true_negatives", LongType(), True),
            StructField("false_positives", LongType(), True),
            StructField("false_negatives", LongType(), True),
            StructField("predicted_positive", LongType(), True),
            StructField("predicted_negative", LongType(), True),
        ]),
    )


def assert_non_trivial(metrics: dict) -> None:
    """T20: probabilities must not be constant and predictions must not be all one class."""
    assert metrics["distinct_probabilities"] > 1, (
        f"Phase 2 stop [{metrics['split']}]: the model produced a constant probability "
        f"({metrics['mean_probability']:.6f}) for every row."
    )
    assert metrics["stddev_probability"] > 0.0, (
        f"Phase 2 stop [{metrics['split']}]: probability standard deviation is zero."
    )
    assert metrics["predicted_positive"] > 0 and metrics["predicted_negative"] > 0, (
        f"Phase 2 stop [{metrics['split']}]: every row was predicted as a single class "
        f"(positive={metrics['predicted_positive']:,}, negative={metrics['predicted_negative']:,}) "
        f"at threshold {RISK_THRESHOLD}."
    )

## 20. MLflow run: training and logging (T17, T18)

One run for the Sprint 2 Random Forest. Autologging is disabled so the run contains exactly the
approved configuration and nothing generated implicitly. Model-registry promotion is out of scope,
so the registry URI is pinned to a local path to stop MLflow resolving a workspace registry.

### Unity Catalog volume required by `mlflow.spark` on serverless

A Spark ML model is a directory of Parquet and metadata written through Hadoop rather than a
single driver-local file, so MLflow needs a distributed staging path to move it to the artifact
store. On serverless and shared clusters the local disk is not visible to the cluster, and MLflow
refuses to guess:

> `UC volume path must be provided to save, log or load SparkML models in Databricks shared or
> serverless clusters.`

The cell below creates a managed volume for that purpose and passes it as `dfs_tmpdir`. The volume
is staging only: the model artifact still lands in the MLflow run, and no modelling data is
written there.

A write-and-delete smoke test runs **before** training. Fitting the forest takes several minutes,
and discovering an unwritable artifact path afterwards means paying for that fit twice.

### Serverless model-cache limits

Serverless caps a single model at 100 MB and all in-memory models in a session at 1 GB. Spark
Connect keeps a fitted model on the driver for as long as any Python object references it, so
repeated fits in one session accumulate until the session fails with
`ML_CACHE_SIZE_OVERFLOW_EXCEPTION`. Two consequences are handled below:

1. The cell before training releases any previously fitted model, including the references held
   by the stored traceback of a cell that raised.
2. Tree training is stopped early by the platform if a model approaches the 100 MB cap, which
   would quietly produce fewer trees than approved. The fitted tree count is asserted against
   `RF_PARAMETERS` immediately after the fit.

If the cache overflows anyway, references stranded in the IPython history cannot be reached from
here. Restart Python (**Detach and re-attach**, or `dbutils.library.restartPython()`) and run the
notebook from the top.

In [0]:
os.environ["MLFLOW_REGISTRY_URI"] = "file:///tmp/mlflow-registry"
mlflow.set_registry_uri("file:///tmp/mlflow-registry")

try:
    import mlflow.pyspark.ml

    mlflow.pyspark.ml.autolog(disable=True)
except Exception as autolog_error:  # noqa: BLE001 - informational only
    print(f"Note: could not disable pyspark.ml autologging ({autolog_error}).")

current_user = spark.sql("SELECT current_user() AS u").collect()[0]["u"]
MLFLOW_EXPERIMENT_PATH = f"/Users/{current_user}/student_attrition_machine_learning"
mlflow.set_experiment(MLFLOW_EXPERIMENT_PATH)
print(f"MLflow experiment: {MLFLOW_EXPERIMENT_PATH}")

In [0]:
MLFLOW_VOLUME = f"{FQ}.mlflow_tmp"
MLFLOW_DFS_TMPDIR = f"/Volumes/{MLFLOW_VOLUME.replace('.', '/')}"

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {MLFLOW_VOLUME} "
    "COMMENT 'Transient staging area required by mlflow.spark on serverless compute. "
    "Holds no modelling data.'"
)

# mlflow.spark reads this when dfs_tmpdir is not passed explicitly; both are set so the path is
# correct however the model is later loaded.
os.environ["MLFLOW_DFS_TMP"] = MLFLOW_DFS_TMPDIR

_smoke_test_path = f"{MLFLOW_DFS_TMPDIR}/_write_check_{uuid.uuid4().hex}"
try:
    spark.range(1).write.mode("overwrite").parquet(_smoke_test_path)
    spark.read.parquet(_smoke_test_path).count()
except Exception as volume_error:
    raise AssertionError(
        f"Phase 2 stop: cannot stage Spark ML artifacts at {MLFLOW_DFS_TMPDIR}. Fix this before "
        f"training, which takes several minutes. Underlying error: {volume_error}"
    ) from volume_error
finally:
    # Cleanup must never mask the real failure above.
    try:
        dbutils.fs.rm(_smoke_test_path, True)
    except Exception as cleanup_error:  # noqa: BLE001 - informational only
        print(f"Note: could not remove the smoke-test path {_smoke_test_path} ({cleanup_error}).")

print(f"MLflow Spark staging volume: {MLFLOW_DFS_TMPDIR} (verified writable and readable)")

In [0]:
# ---- Release any previously fitted model before fitting another ---------------------------------
# Serverless caps in-memory models at 1 GB per session, and Spark Connect keeps a fitted model alive
# on the driver for as long as any Python object references it. Re-running this cell without
# releasing the previous fit accumulates models until the session fails with
# ML_CACHE_SIZE_OVERFLOW_EXCEPTION, which costs a full retrain to discover.
#
# Deleting the obvious variables is not sufficient on its own: a cell that raised keeps its frames
# alive through the stored traceback, and those frames still reference the model.
_released = [
    name for name in ("attrition_model", "fitted_random_forest", "assembled_schema")
    if name in globals()
]
for _name in _released:
    del globals()[_name]

for _attr in ("last_traceback", "last_value", "last_type", "last_exc"):
    try:
        delattr(sys, _attr)
    except (AttributeError, TypeError):
        pass

for _ in range(3):
    gc.collect()
print(f"Released {len(_released)} cached model reference(s) before fitting: {_released or 'none'}")

training_started_at = datetime.now(timezone.utc)

# A previous cell run that failed mid-run can leave a run active, which would make start_run raise.
if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_name=MLFLOW_RUN_NAME) as active_run:
    MLFLOW_RUN_ID = active_run.info.run_id
    MLFLOW_EXPERIMENT_ID = active_run.info.experiment_id

    # ---- Configuration -------------------------------------------------------------------------
    mlflow.log_params({
        "target": TARGET_COLUMN,
        "source_fact_table": FACT_TABLE,
        "source_course_table": COURSE_TABLE,
        "source_teaching_period_table": TEACHING_PERIOD_TABLE,
        "output_table_phase3": PREDICTION_TABLE,
        "split_method": "hash(student_deidentified_hash, seed) % 100 buckets",
        "split_train_fraction": TRAIN_FRACTION,
        "split_validation_fraction": VALIDATION_FRACTION,
        "split_test_fraction": TEST_FRACTION,
        "split_seed": SPLIT_SEED,
        "probability_threshold": RISK_THRESHOLD,
        "feature_count": len(FEATURE_COLUMNS),
        "numeric_feature_count": len(NUMERIC_FEATURES),
        "categorical_feature_count": len(CATEGORICAL_FEATURES),
        "class_weighting_applied": APPLY_CLASS_WEIGHTS,
        "class_weight_formula": "training_rows / (2 * training_rows_in_class)",
        "positive_class_weight": round(POSITIVE_CLASS_WEIGHT, 8),
        "negative_class_weight": round(NEGATIVE_CLASS_WEIGHT, 8),
        "training_started_at": training_started_at.isoformat(),
        **{f"rf_{k}": v for k, v in RF_PARAMETERS.items()},
    })

    # Full manifests go in as artifacts: they are far longer than a parameter value may be.
    mlflow.log_dict(
        {
            "final_feature_list": FEATURE_COLUMNS,
            "numeric_features": NUMERIC_FEATURES,
            "categorical_features": CATEGORICAL_FEATURES,
            "target": TARGET_COLUMN,
            "identifiers_retained_not_features": IDENTIFIER_COLUMNS,
            "prohibited_leakage_columns": PROHIBITED_LEAKAGE_COLUMNS,
        },
        "approved_feature_set.json",
    )
    mlflow.log_dict(
        {"manifest": [dict(m) for m in FEATURE_MANIFEST]},
        "phase1_feature_manifest.json",
    )
    mlflow.log_dict(
        {
            "excluded": [
                {"feature": m["feature"], "source_table": m["source_table"], "reason": m["reason"]}
                for m in FEATURE_MANIFEST if m["decision"] == "exclude"
            ]
        },
        "excluded_features_and_reasons.json",
    )

    # ---- Split evidence ------------------------------------------------------------------------
    for _split_name in ("train", "validation", "test"):
        _stats = SPLIT_STATS[_split_name]
        mlflow.log_params({
            f"{_split_name}_rows": _as_long(_stats["rows"]),
            f"{_split_name}_positive_rows": _as_long(_stats["positive_rows"]),
            f"{_split_name}_negative_rows": _as_long(_stats["negative_rows"]),
        })
        mlflow.log_metric(
            f"{_split_name}_positive_class_pct",
            100.0 * _stats["positive_rows"] / _stats["rows"],
        )

    # ---- Preprocessing contract ----------------------------------------------------------------
    # The fitted Spark model now contains only the Random Forest.
    # Persist the training-derived preprocessing definition separately so the
    # exact transformation can be reconstructed later.

    mlflow.log_dict(
        {
            "numeric_medians": {
                key: float(value)
                for key, value in NUMERIC_MEDIANS.items()
            },
            "categorical_levels": CATEGORY_LEVELS,
            "vector_columns_by_feature": FEATURE_VECTOR_COLUMNS_BY_FEATURE,
            "assembler_input_columns": ASSEMBLER_INPUT_COLUMNS,
            "unknown_category_bucket": True,
            "preprocessing_type": "stateless_manual_onehot",
        },
        "preprocessing_contract.json",
    )

    # ---- Fit on the training split only --------------------------------------------------------

    print("=" * 80)
    print("EFFECTIVE RANDOM FOREST SETTINGS")
    print("=" * 80)

    print(f"numTrees               : {random_forest.getNumTrees()}")
    print(f"maxDepth               : {random_forest.getMaxDepth()}")
    print(f"minInstancesPerNode    : {random_forest.getMinInstancesPerNode()}")
    print(f"featureSubsetStrategy  : {random_forest.getFeatureSubsetStrategy()}")
    print(f"seed                   : {random_forest.getSeed()}")

    assert random_forest.getNumTrees() == RF_NUM_TREES, (
        f"Random Forest numTrees={random_forest.getNumTrees()}, "
        f"expected {RF_NUM_TREES}."
    )

    assert random_forest.getMaxDepth() == RF_MAX_DEPTH, (
        f"Random Forest maxDepth={random_forest.getMaxDepth()}, "
        f"expected {RF_MAX_DEPTH}."
    )

    assert random_forest.getMinInstancesPerNode() == RF_MIN_INSTANCES_PER_NODE, (
        f"Random Forest minInstancesPerNode="
        f"{random_forest.getMinInstancesPerNode()}, "
        f"expected {RF_MIN_INSTANCES_PER_NODE}."
    )

    assert (
        random_forest.getFeatureSubsetStrategy()
        == RF_FEATURE_SUBSET_STRATEGY
    ), (
        f"Random Forest featureSubsetStrategy="
        f"{random_forest.getFeatureSubsetStrategy()}, "
        f"expected {RF_FEATURE_SUBSET_STRATEGY}."
    )

    print("Random Forest parameter verification passed.")
    print("Fitting Random Forest directly on the prepared training split...")

    training_started_at = datetime.now(timezone.utc)

    attrition_model = random_forest.fit(train_prepared_df)

    training_completed_at = datetime.now(timezone.utc)
    training_seconds = (
        training_completed_at - training_started_at
    ).total_seconds()

    mlflow.log_metric(
        "training_duration_seconds",
        training_seconds
    )

    print(f"Training complete in {training_seconds:,.1f}s.")


    # Confirm the fitted forest contains the approved tree count.
    _tree_count_value = attrition_model.getNumTrees

    _fitted_tree_count = int(
        _tree_count_value()
        if callable(_tree_count_value)
        else _tree_count_value
    )

    assert _fitted_tree_count == RF_NUM_TREES, (
        f"Phase 2 stop: fitted forest has {_fitted_tree_count} trees, "
        f"expected {RF_NUM_TREES}."
    )

    mlflow.log_param(
        "fitted_tree_count",
        _fitted_tree_count
    )

    print(
        f"Fitted ensemble confirmed at "
        f"{_fitted_tree_count} trees."
    )
    
    # Logged immediately after fitting rather than at the end of the run, so an artifact-store problem
    # surfaces straight away instead of after the evaluation passes have been paid for.
    mlflow.spark.log_model(
        attrition_model,
        artifact_path="model",
        dfs_tmpdir=MLFLOW_DFS_TMPDIR,
    )
    mlflow.log_param("mlflow_dfs_tmpdir", MLFLOW_DFS_TMPDIR)
    print("Model artifact logged.")

    # ---- Validation evaluation (T19, T20) ------------------------------------------------------
    validation_metrics = evaluate_split(score_split(attrition_model, validation_prepared_df), "validation")
    assert_non_trivial(validation_metrics)
    for _key in ("recall", "accuracy", "precision", "f1", "roc_auc", "mse", "true_positives", "true_negatives",
                 "false_positives", "false_negatives", "predicted_positive", "predicted_negative",
                 "min_probability", "max_probability", "mean_probability", "stddev_probability"):
        mlflow.log_metric(f"validation_{_key}", float(validation_metrics[_key]))
    print(f"Validation recall (primary): {validation_metrics['recall']:.6f}")

    # ---- Test evaluation on the untouched split (T21) ------------------------------------------
    # Reached only after the validation checks above pass. The model is not altered in between.
    test_metrics = evaluate_split(score_split(attrition_model, test_prepared_df), "test")
    assert_non_trivial(test_metrics)
    for _key in ("recall", "accuracy", "precision", "f1", "roc_auc", "true_positives", "true_negatives",
                 "false_positives", "false_negatives", "predicted_positive", "predicted_negative",
                 "min_probability", "max_probability", "mean_probability", "stddev_probability"):
        mlflow.log_metric(f"test_{_key}", float(test_metrics[_key]))
    print(f"Test recall (primary): {test_metrics['recall']:.6f}")

    mlflow.set_tags({
        "sprint": "2",
        "phase": "2",
        "target": TARGET_COLUMN,
        "primary_metric": "recall",
        "model_family": "spark_ml_random_forest",
    })

print(f"MLflow experiment id : {MLFLOW_EXPERIMENT_ID}")
print(f"MLflow run id        : {MLFLOW_RUN_ID}")

## 21. Validation and test results (T19, T21)

In [0]:
display(metrics_to_dataframe([validation_metrics, test_metrics]))

### 21.1 Probability distribution and non-triviality evidence (T20)

In [0]:
probability_diagnostics_df = spark.createDataFrame(
    [
        (
            m["split"],
            round(m["min_probability"], 6),
            round(m["max_probability"], 6),
            round(m["mean_probability"], 6),
            round(m["stddev_probability"], 6),
            _as_long(m["distinct_probabilities"]),
            _as_long(m["predicted_positive"]),
            _as_long(m["predicted_negative"]),
        )
        for m in (validation_metrics, test_metrics)
    ],
    schema=StructType([
        StructField("split", StringType(), False),
        StructField("min_probability", DoubleType(), True),
        StructField("max_probability", DoubleType(), True),
        StructField("mean_probability", DoubleType(), True),
        StructField("stddev_probability", DoubleType(), True),
        StructField("distinct_probabilities", LongType(), True),
        StructField("predicted_positive", LongType(), True),
        StructField("predicted_negative", LongType(), True),
    ]),
)
display(probability_diagnostics_df)

print(
    "Non-triviality confirmed on both splits: probabilities vary and both classes are predicted at "
    f"threshold {RISK_THRESHOLD}."
)

### 21.2 Feature importances

Reported for reviewer attention only. No feature is added, removed, or reweighted on the basis of
these values; that would be the automated feature selection the plan prohibits.

Importances are produced per assembled vector slot, so one-hot slots are summed back to the
approved feature they came from. The `vector_slots` column shows how many slots each feature
occupies: one for a numeric feature, and one per category plus one reserved bucket for unseen and
null values for a categorical feature.

In [0]:
raw_importances = attrition_model.featureImportances.toArray()

assert len(raw_importances) == len(ASSEMBLER_INPUT_COLUMNS), (
    f"Phase 2 stop: Random Forest importance width "
    f"{len(raw_importances)} != assembler width "
    f"{len(ASSEMBLER_INPUT_COLUMNS)}."
)

slot_index = {
    column: index
    for index, column in enumerate(ASSEMBLER_INPUT_COLUMNS)
}

importance_by_feature = {}
slots_by_feature = {}

for feature in FEATURE_COLUMNS:
    feature_columns = FEATURE_VECTOR_COLUMNS_BY_FEATURE[feature]

    importance_by_feature[feature] = float(
        sum(
            raw_importances[slot_index[column]]
            for column in feature_columns
        )
    )

    slots_by_feature[feature] = len(feature_columns)


assert abs(sum(importance_by_feature.values()) - 1.0) < 1e-6, (
    "Phase 2 stop: aggregated feature importances "
    "do not sum to approximately 1.0."
)


importance_df = spark.createDataFrame(
    [
        (
            feature,
            "numeric" if feature in NUMERIC_FEATURES else "categorical",
            int(slots_by_feature[feature]),
            round(importance, 8),
            round(importance * 100.0, 5),
        )
        for feature, importance in importance_by_feature.items()
    ],
    schema=StructType([
        StructField("feature", StringType(), False),
        StructField("role", StringType(), False),
        StructField("vector_slots", LongType(), False),
        StructField("importance", DoubleType(), True),
        StructField("importance_pct", DoubleType(), True),
    ]),
)

display(
    importance_df.orderBy(
        F.col("importance").desc()
    )
)

print(f"Numeric slots      : {len(NUMERIC_FEATURES)}")
print(
    f"Categorical slots  : "
    f"{sum(slots_by_feature[f] for f in CATEGORICAL_FEATURES)}"
)
print(f"Total vector width : {len(raw_importances)}")
print(
    f"Importance total   : "
    f"{sum(importance_by_feature.values()):.8f}"
)

# Add feature-importance evidence to the existing MLflow training run.
if (
    mlflow.active_run() is not None
    and mlflow.active_run().info.run_id != MLFLOW_RUN_ID
):
    mlflow.end_run()

with mlflow.start_run(run_id=MLFLOW_RUN_ID):
    mlflow.log_dict(
        {
            "feature_importances": importance_by_feature,
            "vector_slots_per_feature": {
                key: int(value)
                for key, value in slots_by_feature.items()
            },
            "categorical_training_levels": {
                feature: len(CATEGORY_LEVELS[feature])
                for feature in CATEGORICAL_FEATURES
            },
            "unknown_bucket_per_categorical_feature": True,
        },
        "feature_importances.json",
    )

    mlflow.log_param(
        "assembled_vector_width",
        len(raw_importances),
    )

print(
    f"Feature importances recorded for "
    f"{len(importance_by_feature)} original features."
)

## 22. Phase 2 summary

Run this cell for the figures the human review needs. Interpret the metrics against limitations
L-1 to L-3 in `plan.md`: ROC-AUC near 0.5 and precision near the 5.5% base rate are the expected,
correct outcome for this synthetic dataset, not a fault in the workflow.

In [0]:
print("=" * 96)
print("PHASE 2 SUMMARY - split, preprocessing, Random Forest, MLflow, evaluation")
print("=" * 96)
print(f"Approved feature manifest    : {len(FEATURE_COLUMNS)} features used unchanged from Phase 1")
print(f"  numeric                    : {len(NUMERIC_FEATURES)}")
print(f"  categorical                : {len(CATEGORICAL_FEATURES)}")
print(f"Assembled vector width       : {len(raw_importances)} slots")
print(f"Target                       : {TARGET_COLUMN}")
print()
print("Split (hash of student_deidentified_hash with seed 42, 100 buckets)")
for _split_name in ("train", "validation", "test"):
    _s = SPLIT_STATS[_split_name]
    print(
        f"  {_split_name:<11}: {_as_long(_s['rows']):>8,} rows "
        f"({100.0 * _s['rows'] / total_supervised_rows:5.2f}%)  "
        f"positive {_as_long(_s['positive_rows']):>6,} "
        f"({100.0 * _s['positive_rows'] / _s['rows']:.4f}%)"
    )
print("  students in more than one split: 0")
print()
print("Preprocessing")
print("  numeric     : cast to double, training-derived median replacement using Spark expressions")
print("  categorical : training-derived deterministic manual one-hot columns + unseen/NULL bucket")
print("  fitting     : no Spark ML preprocessing Estimator fitted")
print("  assembly    : VectorAssembler.transform() only")
print("  scaling     : not applied (not required by the approved Random Forest)")
print(
    f"  class weights applied: {APPLY_CLASS_WEIGHTS} "
    f"(positive {POSITIVE_CLASS_WEIGHT:.6f}, negative {NEGATIVE_CLASS_WEIGHT:.6f})"
)
print()
print("Random Forest")
for _name, _value in RF_PARAMETERS.items():
    print(f"  {_name:<24}: {_value}")
print()
print(f"Metrics at threshold {RISK_THRESHOLD} (recall is the primary metric, positive class)")
for _m in (validation_metrics, test_metrics):
    print(f"  {_m['split']:<11}: recall {_m['recall']:.6f}  accuracy {_m['accuracy']:.6f}  "
          f"precision {_m['precision']:.6f}  f1 {_m['f1']:.6f}  roc_auc {_m['roc_auc']:.6f}")
    print(f"               TP {_m['true_positives']:,}  TN {_m['true_negatives']:,}  "
          f"FP {_m['false_positives']:,}  FN {_m['false_negatives']:,}")
    print(f"               predicted positive {_m['predicted_positive']:,}  "
          f"predicted negative {_m['predicted_negative']:,}")
print()
print("Non-triviality (T20)")
for _m in (validation_metrics, test_metrics):
    print(f"  {_m['split']:<11}: {_m['distinct_probabilities']:,} distinct probabilities, "
          f"stddev {_m['stddev_probability']:.6f}, range "
          f"[{_m['min_probability']:.6f}, {_m['max_probability']:.6f}]")
print()
print(f"MLflow experiment : {MLFLOW_EXPERIMENT_PATH}")
print(f"MLflow experiment id : {MLFLOW_EXPERIMENT_ID}")
print(f"MLflow run id        : {MLFLOW_RUN_ID}   <-- carried into Phase 3")
print()
print("Delta tables written by Phase 2: none.")
print("=" * 96)

## 23. Phase 2 human review outcome - approved 2026-08-06

The human reviewer approved Phase 2, including material refinement M-1 below, and authorised
persistence. Tasks **T9-T21** are marked complete in `tasks.md`.

### Routine corrections reported at this gate

| Correction | Reason |
|---|---|
| Recall, precision, and F1 reported for the positive class rather than as weighted averages | A weighted average over a 94.5% negative class is dominated by the majority and would misstate the primary metric |
| Predicted class derived from the 0.50 threshold on the positive probability rather than read from the classifier's `prediction` column | Makes the reported metrics identical to those the Phase 3 risk flag will reproduce |
| One-hot importance slots summed back to their approved feature | Reporting only; no feature is added, removed, or reweighted |
| MLflow registry URI pinned to a local path and autologging disabled | Registry promotion is out of scope; the run holds exactly the approved configuration |
| A Unity Catalog volume `workspace.student_aggregate.mlflow_tmp` is created and passed as `dfs_tmpdir`, verified writable before training | Reported after the gate, during first execution. `mlflow.spark` cannot log a Spark ML model on serverless without a UC volume staging path and fails with `UC volume path must be provided to save, log or load SparkML models`. The volume holds no modelling data and changes no approved behaviour; the pre-training smoke test exists because the fit takes several minutes and this failure previously surfaced only after paying for it |

### Material refinement - approved, recorded as amendment A-4 in `plan.md`

**M-1: split assignment by hashing the student identifier instead of `randomSplit`.**

- **Change:** assign splits with `pmod(hash(student_deidentified_hash, 42), 100)` into buckets
  0-69 train, 70-84 validation, 85-99 test, rather than calling `randomSplit([0.7, 0.15, 0.15],
  seed=42)`. Ratios, seed, determinism, and disjointness are all as approved; only *which* rows
  land in each split differs.
- **Reason:** `randomSplit` samples per partition, so it is stable only while the physical plan and
  partitioning are stable. Spark's documented mitigation is to cache or save the DataFrame before
  splitting, which serverless compute forbids. Because the modelling DataFrame is recomputed on
  every action, a replan between the fit action and an evaluation action could move rows between
  splits and leak test data into training, silently.
- **Evidence:** the serverless failure is reproducible - `cache()` raises
  `NOT_SUPPORTED_WITH_SERVERLESS: PERSIST TABLE is not supported`. Hashing is a pure function of the
  data, so section 14.2 can and does prove that no student appears in more than one split.
- **Affected artefacts:** `plan.md` section 6, tasks T9 and T10.
- **Impact:** none on the target, grain, features, model parameters, threshold, or output schema.
  Split membership changes, so metrics will differ slightly from a `randomSplit` run.

Approved on 2026-08-06. `plan.md` section 6 now records the hash-bucket method, and the full
amendment is recorded as A-4.

# Phase 3: Inference, risk flag generation, Delta persistence, retrieval validation

Implements tasks **T22-T29**. Runs only after Phase 2 approval, which authorised persistence.

Preprocessing reuses the training-derived medians and categorical levels; no feature is added or substituted. The `attrition_model`
fitted in section 20 is applied exactly as trained. The only Delta table written by this notebook
is the approved prediction table, in overwrite mode. No source table is modified.

## 24. Applicable-student inference dataset (T22)

The approved inference population is **every valid student row in the Phase 1 canonical modelling
projection**, per `plan.md` section 10 ("apply the fitted pipeline to every valid student row") and
clarified decision 1 ("score every valid student row in the prepared modelling dataset").

This is the whole modelling dataset, not the test split. Three consequences worth stating plainly:

1. Train, validation, and test rows are all scored, because the application needs a current risk
   result for every student rather than only for the held-out sample. Rows seen during training are
   scored optimistically; that affects the stored percentages, not the Phase 2 metrics, which were
   measured on untouched data.
2. Rows whose outcome is unknown are scored too. They were excluded from training because there is
   nothing to learn from them, but they are exactly the students the application most needs a
   prediction for.
3. Preprocessing uses the medians and categorical levels computed from the training split only. Unseen categories fall into
   the reserved bucket and missing numerics take the training median, so no inference row can
   influence the transformation applied to it.

The same `prepare_feature_vector` function used in Phase 2 is reused here, so inference receives an
identically constructed feature vector.

**This cell triggers no Spark job.** The population size, uniqueness, and outcome breakdown were
all measured and asserted in Phase 1 section 10 against this exact DataFrame, so recounting them
here would cost a full pass over the eleven-join chain to rediscover validated figures. The
expected row count is carried forward from Phase 1 and the actual count is checked against it in
section 27, which makes that check stronger than a self-consistent recount.

In [0]:
INFERENCE_POPULATION_DESCRIPTION = (
    "every valid student row in the approved Phase 1 canonical modelling projection"
)

# prepare_feature_vector recomputes the training-derived preprocessing columns and reassembles the
# feature vector; the label and weight columns used during training are not required at inference time.
inference_input_df = prepare_feature_vector(
    phase1_modelling_df
)

# Carried forward from the Phase 1 assertions in section 10, not recomputed.
INFERENCE_POPULATION_ROWS = modelling_rows
INFERENCE_POPULATION_STUDENTS = modelling_students
INFERENCE_KNOWN_OUTCOME_ROWS = supervised_rows
INFERENCE_UNKNOWN_OUTCOME_ROWS = unsupervised_rows

assert INFERENCE_POPULATION_ROWS == INFERENCE_POPULATION_STUDENTS, (
    f"Phase 3 stop: the inference population is not one row per student "
    f"({INFERENCE_POPULATION_ROWS:,} rows vs {INFERENCE_POPULATION_STUDENTS:,} students)."
)
assert STUDENT_ID_COLUMN in inference_input_df.columns, (
    f"Phase 3 stop: {STUDENT_ID_COLUMN} is missing from the inference input."
)

print(f"Inference population: {INFERENCE_POPULATION_DESCRIPTION}")
print(f"  rows                 : {INFERENCE_POPULATION_ROWS:,}  (asserted in Phase 1 section 10)")
print(f"  distinct students    : {INFERENCE_POPULATION_STUDENTS:,}")
print(f"  known outcome rows   : {INFERENCE_KNOWN_OUTCOME_ROWS:,} (used by Phase 2 splits)")
print(f"  unknown outcome rows : {INFERENCE_UNKNOWN_OUTCOME_ROWS:,} (never trainable, still scored)")

## 25. Scoring, risk percentage, and risk flag (T23, T24, T25)

- `attrition_risk_probability` - the positive-class probability, kept as an unrounded intermediate.
- `attrition_risk_percentage` - the probability multiplied by 100.
- `attrition_risk_flag` - the threshold applied to the **unrounded probability**, so a probability
  of exactly 0.50 is true and 0.49 is false. The flag is never derived from the rounded percentage.

In [0]:
PROBABILITY_OUTPUT_COLUMN = "attrition_risk_probability"
PERCENTAGE_OUTPUT_COLUMN = "attrition_risk_percentage"
FLAG_OUTPUT_COLUMN = "attrition_risk_flag"
THRESHOLD_OUTPUT_COLUMN = "prediction_threshold"
RUN_ID_OUTPUT_COLUMN = "mlflow_run_id"
SCORED_AT_OUTPUT_COLUMN = "scored_at"

scored_at_timestamp = datetime.now(timezone.utc)

scored_inference_df = (
    attrition_model.transform(inference_input_df)
    .withColumn(PROBABILITY_OUTPUT_COLUMN, vector_to_array(F.col(PROBABILITY_COLUMN))[1])
    .withColumn(PERCENTAGE_OUTPUT_COLUMN, F.col(PROBABILITY_OUTPUT_COLUMN) * F.lit(100.0))
    .withColumn(FLAG_OUTPUT_COLUMN, F.col(PROBABILITY_OUTPUT_COLUMN) >= F.lit(RISK_THRESHOLD))
    .withColumn(THRESHOLD_OUTPUT_COLUMN, F.lit(RISK_THRESHOLD).cast("double"))
    .withColumn(RUN_ID_OUTPUT_COLUMN, F.lit(MLFLOW_RUN_ID))
    .withColumn(SCORED_AT_OUTPUT_COLUMN, F.lit(scored_at_timestamp).cast("timestamp"))
)

print(f"Scored at         : {scored_at_timestamp.isoformat()}")
print(f"Threshold applied : {RISK_THRESHOLD} (>= is true, < is false, applied before any rounding)")
print(f"MLflow run linked : {MLFLOW_RUN_ID}")

## 26. Approved prediction output and scoring staging table (T26)

The approved output is exactly the six columns from `plan.md` section 10, in the approved order.

### Why the scoring result is staged

The acceptance criteria require that every check passes **before** anything is persisted to the
approved table. Written naively that forces the forest to run twice over the whole population,
once to compute the validation aggregates and again to produce the rows being written, because
serverless compute forbids `cache()` and Spark therefore recomputes the entire eleven-join chain
and all 100 trees on every action.

Instead the scored result is materialised **once** into a transient staging table, all validation
runs against that table as cheap columnar Delta scans, and the approved table is replaced only
after every check has passed. The staging table is dropped either way. This is the mitigation
`plan.md` section 15 already prescribes for the serverless caching ban: where a stage needs a
reusable materialisation, write Delta rather than cache.

The guarantee is unchanged and arguably stronger. Nothing reaches
`student_attrition_risk_prediction` until validation has passed, and validation now inspects
bytes that survived a round trip through Delta rather than a plan that would be recomputed.

### What staging carries that the approved table does not

Staging holds the unrounded `attrition_risk_probability` so section 27 can prove the flag was
derived from the probability rather than from the rounded percentage. That column is dropped when
the approved table is published, because widening the stored schema would need human approval and
an SDD amendment.

In [0]:
PREDICTION_OUTPUT_COLUMNS = [
    STUDENT_ID_COLUMN,
    PERCENTAGE_OUTPUT_COLUMN,
    FLAG_OUTPUT_COLUMN,
    THRESHOLD_OUTPUT_COLUMN,
    RUN_ID_OUTPUT_COLUMN,
    SCORED_AT_OUTPUT_COLUMN,
]

# Transient. Written in section 27, dropped on the failure path in 27.1 and on the success path in 28.
STAGING_TABLE = f"{PREDICTION_TABLE}__scoring_staging"

STAGING_COLUMNS = PREDICTION_OUTPUT_COLUMNS + [PROBABILITY_OUTPUT_COLUMN]

staging_output_df = scored_inference_df.select(*STAGING_COLUMNS)

print(f"Approved persisted schema ({len(PREDICTION_OUTPUT_COLUMNS)} columns):")
for _column in PREDICTION_OUTPUT_COLUMNS:
    print(f"  {_column}")
print()
print(f"Staging table : {STAGING_TABLE}")
print(f"  additionally carries {PROBABILITY_OUTPUT_COLUMN} for validation, dropped at publish")

## 27. Materialise the scoring result and validate it (T23, T27)

The write below is the **only** time the forest is applied to the full population. Everything
after it reads Delta.

In [0]:
(
    staging_output_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(STAGING_TABLE)
)

staged_df = spark.table(STAGING_TABLE)
print(f"Scoring materialised to {STAGING_TABLE}. Validation below reads this table, not the model.")

### 27.1 Pre-publish output validation

Every check the Phase 3 acceptance criteria require, computed in a single scan of the staging
table. **The approved table is not touched unless all of them pass.**

In [0]:
output_checks = staged_df.agg(
    F.count(F.lit(1)).alias("rows"),
    F.countDistinct(F.col(STUDENT_ID_COLUMN)).alias("distinct_students"),
    F.count(F.when(F.col(STUDENT_ID_COLUMN).isNull(), F.lit(1))).alias("null_students"),
    F.count(F.when(F.col(PROBABILITY_OUTPUT_COLUMN).isNull(), F.lit(1))).alias("null_probabilities"),
    F.count(
        F.when(
            (F.col(PROBABILITY_OUTPUT_COLUMN) < F.lit(0.0)) | (F.col(PROBABILITY_OUTPUT_COLUMN) > F.lit(1.0)),
            F.lit(1),
        )
    ).alias("probabilities_out_of_range"),
    F.count(F.when(F.col(PERCENTAGE_OUTPUT_COLUMN).isNull(), F.lit(1))).alias("null_percentages"),
    F.count(
        F.when(
            (F.col(PERCENTAGE_OUTPUT_COLUMN) < F.lit(0.0)) | (F.col(PERCENTAGE_OUTPUT_COLUMN) > F.lit(100.0)),
            F.lit(1),
        )
    ).alias("percentages_out_of_range"),
    F.count(F.when(F.col(FLAG_OUTPUT_COLUMN).isNull(), F.lit(1))).alias("null_flags"),
    F.count(F.when(F.col(THRESHOLD_OUTPUT_COLUMN) != F.lit(RISK_THRESHOLD), F.lit(1))).alias("wrong_threshold"),
    F.count(
        F.when(
            F.col(RUN_ID_OUTPUT_COLUMN).isNull() | (F.length(F.col(RUN_ID_OUTPUT_COLUMN)) == F.lit(0)),
            F.lit(1),
        )
    ).alias("missing_run_id"),
    F.count(F.when(F.col(SCORED_AT_OUTPUT_COLUMN).isNull(), F.lit(1))).alias("null_scored_at"),
    # Proves the flag was derived from the unrounded probability and not from the percentage.
    F.count(
        F.when(
            F.col(FLAG_OUTPUT_COLUMN) != (F.col(PROBABILITY_OUTPUT_COLUMN) >= F.lit(RISK_THRESHOLD)),
            F.lit(1),
        )
    ).alias("flag_threshold_mismatches"),
    F.count(F.when(F.col(FLAG_OUTPUT_COLUMN), F.lit(1))).alias("flag_true"),
    F.count(F.when(~F.col(FLAG_OUTPUT_COLUMN), F.lit(1))).alias("flag_false"),
    F.min(F.col(PROBABILITY_OUTPUT_COLUMN)).alias("min_probability"),
    F.max(F.col(PROBABILITY_OUTPUT_COLUMN)).alias("max_probability"),
    F.avg(F.col(PROBABILITY_OUTPUT_COLUMN)).alias("mean_probability"),
).collect()[0]

OUTPUT_ROWS = _as_long(output_checks["rows"])

# (check name, observed value, expected value)
OUTPUT_VALIDATIONS = [
    ("rows match the Phase 1 validated population", OUTPUT_ROWS, INFERENCE_POPULATION_ROWS),
    ("one row per student", _as_long(output_checks["distinct_students"]), OUTPUT_ROWS),
    ("no null student identifiers", _as_long(output_checks["null_students"]), 0),
    ("no null intermediate probabilities", _as_long(output_checks["null_probabilities"]), 0),
    ("probabilities within [0, 1]", _as_long(output_checks["probabilities_out_of_range"]), 0),
    ("no null percentages", _as_long(output_checks["null_percentages"]), 0),
    ("percentages within [0, 100]", _as_long(output_checks["percentages_out_of_range"]), 0),
    ("no null flags", _as_long(output_checks["null_flags"]), 0),
    ("threshold is exactly 0.50 on every row", _as_long(output_checks["wrong_threshold"]), 0),
    ("MLflow run id populated on every row", _as_long(output_checks["missing_run_id"]), 0),
    ("scored_at populated on every row", _as_long(output_checks["null_scored_at"]), 0),
    ("flag agrees with the unrounded threshold rule", _as_long(output_checks["flag_threshold_mismatches"]), 0),
]

output_validation_df = spark.createDataFrame(
    [
        (name, _as_long(observed), _as_long(expected), "PASS" if observed == expected else "FAIL")
        for name, observed, expected in OUTPUT_VALIDATIONS
    ],
    schema=StructType([
        StructField("check", StringType(), False),
        StructField("observed", LongType(), True),
        StructField("expected", LongType(), True),
        StructField("result", StringType(), False),
    ]),
)
display(output_validation_df)

output_failures = [(name, observed, expected) for name, observed, expected in OUTPUT_VALIDATIONS if observed != expected]
if output_failures:
    spark.sql(f"DROP TABLE IF EXISTS {STAGING_TABLE}")
    raise AssertionError(
        f"Phase 3 stop: output validation failed. {PREDICTION_TABLE} was not touched and the staging "
        f"table has been dropped. Failures: {output_failures}"
    )

print(f"All {len(OUTPUT_VALIDATIONS)} output validations passed. Safe to publish.")
print(f"  flag true  : {_as_long(output_checks['flag_true']):,} "
      f"({100.0 * output_checks['flag_true'] / OUTPUT_ROWS:.4f}%)")
print(f"  flag false : {_as_long(output_checks['flag_false']):,} "
      f"({100.0 * output_checks['flag_false'] / OUTPUT_ROWS:.4f}%)")
print(f"  probability range: [{output_checks['min_probability']:.6f}, "
      f"{output_checks['max_probability']:.6f}], mean {output_checks['mean_probability']:.6f}")

### 27.2 Capture the retrieval expectations

Two flagged and two unflagged students are sampled from the validated staging table, so the
retrieval test in section 29.4 exercises both branches of the application lookup path. Their
expected values are collected now, while staging still exists, which means section 29.4 can prove
the published table matches what was scored without rescoring anything.

In [0]:
retrieval_sample_rows = (
    staged_df.filter(F.col(FLAG_OUTPUT_COLUMN)).select(*STAGING_COLUMNS).limit(2).collect()
    + staged_df.filter(~F.col(FLAG_OUTPUT_COLUMN)).select(*STAGING_COLUMNS).limit(2).collect()
)
assert retrieval_sample_rows, "Phase 3 stop: no students available to test retrieval."

expected_by_student = {row[STUDENT_ID_COLUMN]: row.asDict() for row in retrieval_sample_rows}
retrieval_sample_keys = list(expected_by_student)

_sampled_flagged = sum(1 for row in expected_by_student.values() if row[FLAG_OUTPUT_COLUMN])
print(f"Retrieval expectations captured for {len(retrieval_sample_keys)} students: "
      f"{_sampled_flagged} flagged, {len(retrieval_sample_keys) - _sampled_flagged} not flagged.")

## 28. Publish to the approved Delta table and drop staging (T28)

Replaces the approved prediction table with the validated snapshot, projected down to the six
approved columns. This is a Delta-to-Delta copy of six narrow columns, so it costs a columnar
scan rather than another pass through the model.

Reached only because every check in section 27.1 passed. No source table is read for write or
modified, and the transient staging table is dropped afterwards.

In [0]:
assert not output_failures, "Phase 3 stop: refusing to publish after a failed validation."

_source_tables = {entry["fqn"] for entry in SOURCE_INVENTORY}
for _target in (PREDICTION_TABLE, STAGING_TABLE):
    assert _target not in _source_tables, (
        f"Phase 3 stop: the write target {_target} is a source table."
    )

(
    staged_df.select(*PREDICTION_OUTPUT_COLUMNS).write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(PREDICTION_TABLE)
)

print(f"Published: {PREDICTION_TABLE} ({OUTPUT_ROWS:,} rows, overwrite mode)")

spark.sql(f"DROP TABLE IF EXISTS {STAGING_TABLE}")
print(f"Dropped staging table: {STAGING_TABLE}")

## 29. Post-publish verification and application retrieval (T29)

Everything below queries `student_attrition_risk_prediction` itself, so it verifies the table the
application will actually read rather than the staging table it was copied from. These are narrow
columnar scans over six columns and cost a small fraction of the scoring pass.

### 29.1 Table exists, is queryable, and has the approved schema

In [0]:
assert spark.catalog.tableExists(PREDICTION_TABLE), (
    f"Phase 3 stop: {PREDICTION_TABLE} does not exist after publication."
)
assert not spark.catalog.tableExists(STAGING_TABLE), (
    f"Phase 3 stop: the transient staging table {STAGING_TABLE} was not dropped."
)

persisted_df = spark.table(PREDICTION_TABLE)

persisted_schema_df = spark.createDataFrame(
    [
        (position, field.name, field.dataType.simpleString(), field.nullable)
        for position, field in enumerate(persisted_df.schema.fields, start=1)
    ],
    schema=StructType([
        StructField("position", LongType(), False),
        StructField("column", StringType(), False),
        StructField("data_type", StringType(), False),
        StructField("nullable", BooleanType(), False),
    ]),
)
display(persisted_schema_df)

assert persisted_df.columns == PREDICTION_OUTPUT_COLUMNS, (
    f"Phase 3 stop: persisted schema {persisted_df.columns} does not match the approved schema "
    f"{PREDICTION_OUTPUT_COLUMNS}."
)

### 29.2 Persisted row count, uniqueness, and value integrity

In [0]:
persisted_checks = persisted_df.agg(
    F.count(F.lit(1)).alias("rows"),
    F.countDistinct(F.col(STUDENT_ID_COLUMN)).alias("distinct_students"),
    F.count(F.when(F.col(STUDENT_ID_COLUMN).isNull(), F.lit(1))).alias("null_students"),
    F.count(
        F.when(
            (F.col(PERCENTAGE_OUTPUT_COLUMN) < F.lit(0.0)) | (F.col(PERCENTAGE_OUTPUT_COLUMN) > F.lit(100.0)),
            F.lit(1),
        )
    ).alias("percentages_out_of_range"),
    F.countDistinct(F.col(THRESHOLD_OUTPUT_COLUMN)).alias("distinct_thresholds"),
    F.min(F.col(THRESHOLD_OUTPUT_COLUMN)).alias("threshold_value"),
    F.countDistinct(F.col(RUN_ID_OUTPUT_COLUMN)).alias("distinct_run_ids"),
    F.countDistinct(F.col(SCORED_AT_OUTPUT_COLUMN)).alias("distinct_scored_at"),
    F.count(F.when(F.col(FLAG_OUTPUT_COLUMN), F.lit(1))).alias("flag_true"),
    F.count(F.when(~F.col(FLAG_OUTPUT_COLUMN), F.lit(1))).alias("flag_false"),
).collect()[0]

PERSISTED_ROWS = _as_long(persisted_checks["rows"])
PERSISTED_STUDENTS = _as_long(persisted_checks["distinct_students"])
PERSISTED_FLAG_TRUE = _as_long(persisted_checks["flag_true"])
PERSISTED_FLAG_FALSE = _as_long(persisted_checks["flag_false"])

PERSISTED_VALIDATIONS = [
    ("published rows match the validated scoring result", PERSISTED_ROWS, OUTPUT_ROWS),
    ("published rows match the Phase 1 validated population", PERSISTED_ROWS, INFERENCE_POPULATION_ROWS),
    ("one row per student", PERSISTED_STUDENTS, PERSISTED_ROWS),
    ("no null student identifiers", _as_long(persisted_checks["null_students"]), 0),
    ("percentages within [0, 100]", _as_long(persisted_checks["percentages_out_of_range"]), 0),
    ("a single threshold value", _as_long(persisted_checks["distinct_thresholds"]), 1),
    ("a single MLflow run id", _as_long(persisted_checks["distinct_run_ids"]), 1),
    ("a single scored_at timestamp", _as_long(persisted_checks["distinct_scored_at"]), 1),
    ("flags account for every row", PERSISTED_FLAG_TRUE + PERSISTED_FLAG_FALSE, PERSISTED_ROWS),
]

persisted_validation_df = spark.createDataFrame(
    [
        (name, _as_long(observed), _as_long(expected), "PASS" if observed == expected else "FAIL")
        for name, observed, expected in PERSISTED_VALIDATIONS
    ],
    schema=StructType([
        StructField("check", StringType(), False),
        StructField("observed", LongType(), True),
        StructField("expected", LongType(), True),
        StructField("result", StringType(), False),
    ]),
)
display(persisted_validation_df)

persisted_failures = [(n, o, e) for n, o, e in PERSISTED_VALIDATIONS if o != e]
assert not persisted_failures, f"Phase 3 stop: post-publish validation failed -> {persisted_failures}"

assert float(persisted_checks["threshold_value"]) == RISK_THRESHOLD, (
    f"Phase 3 stop: persisted threshold {persisted_checks['threshold_value']} is not {RISK_THRESHOLD}."
)

persisted_run_id = persisted_df.select(RUN_ID_OUTPUT_COLUMN).limit(1).collect()[0][RUN_ID_OUTPUT_COLUMN]
assert persisted_run_id == MLFLOW_RUN_ID, (
    f"Phase 3 stop: persisted MLflow run id {persisted_run_id} does not match the Phase 2 training run "
    f"{MLFLOW_RUN_ID}."
)

print(f"Post-publish validation passed. {PERSISTED_ROWS:,} rows, {PERSISTED_STUDENTS:,} unique students.")
print(f"  MLflow run id matches the Phase 2 training run: {persisted_run_id}")

### 29.3 Risk percentage distribution and sample

Read from the persisted table, both because that is what the application will see and because
rescoring the whole population through a 100-tree forest is expensive enough not to repeat when a
cheap Delta scan answers the same question.

Read the spread against limitation L-3: class weighting recentres the scores on 50, so these are
relative ranking values rather than calibrated likelihoods of attrition.

In [0]:
percentage_distribution_df = (
    persisted_df.withColumn(
        "percentage_band_floor",
        (F.floor(F.col(PERCENTAGE_OUTPUT_COLUMN) / F.lit(5)) * F.lit(5)).cast("int"),
    )
    .groupBy("percentage_band_floor")
    .agg(
        F.count(F.lit(1)).alias("students"),
        F.count(F.when(F.col(FLAG_OUTPUT_COLUMN), F.lit(1))).alias("flagged"),
    )
    .withColumn(
        "percentage_band",
        F.concat_ws(
            "",
            F.col("percentage_band_floor").cast("string"),
            F.lit(" to "),
            (F.col("percentage_band_floor") + F.lit(5)).cast("string"),
        ),
    )
    .withColumn("share_pct", F.round(100.0 * F.col("students") / F.lit(PERSISTED_ROWS), 4))
    .orderBy(F.col("percentage_band_floor"))
    .select("percentage_band", "students", "share_pct", "flagged")
)
display(percentage_distribution_df)

In [0]:
display(persisted_df.orderBy(F.col(PERCENTAGE_OUTPUT_COLUMN).desc()).limit(20))

### 29.4 Application retrieval by `student_deidentified_hash`

Reproduces the lookup the Databricks application performs: a point query on the logical primary
key, checked field by field against the values captured from the validated scoring result in
section 27.2. Because those expectations were taken before publication, this also proves the
staging-to-approved copy preserved every value exactly.

In [0]:
retrieval_rows = []
for student_key in retrieval_sample_keys:
    retrieved = persisted_df.filter(F.col(STUDENT_ID_COLUMN) == F.lit(student_key)).collect()
    assert len(retrieved) == 1, (
        f"Phase 3 stop: retrieval by {STUDENT_ID_COLUMN}={student_key} returned {len(retrieved)} rows, expected 1."
    )
    retrieved_row = retrieved[0].asDict()
    expected_row = expected_by_student[student_key]

    field_checks = {
        PERCENTAGE_OUTPUT_COLUMN: retrieved_row[PERCENTAGE_OUTPUT_COLUMN] == expected_row[PERCENTAGE_OUTPUT_COLUMN],
        FLAG_OUTPUT_COLUMN: retrieved_row[FLAG_OUTPUT_COLUMN] == expected_row[FLAG_OUTPUT_COLUMN],
        THRESHOLD_OUTPUT_COLUMN: retrieved_row[THRESHOLD_OUTPUT_COLUMN] == RISK_THRESHOLD,
        RUN_ID_OUTPUT_COLUMN: retrieved_row[RUN_ID_OUTPUT_COLUMN] == MLFLOW_RUN_ID,
        SCORED_AT_OUTPUT_COLUMN: retrieved_row[SCORED_AT_OUTPUT_COLUMN] == expected_row[SCORED_AT_OUTPUT_COLUMN],
        # The published flag still agrees with the unrounded probability held only in staging.
        "flag_matches_unrounded_probability": (
            retrieved_row[FLAG_OUTPUT_COLUMN]
            == (expected_row[PROBABILITY_OUTPUT_COLUMN] >= RISK_THRESHOLD)
        ),
    }
    mismatched = [name for name, ok in field_checks.items() if not ok]
    assert not mismatched, (
        f"Phase 3 stop: retrieved values for {student_key} do not match what was scored -> {mismatched}"
    )

    retrieval_rows.append((
        student_key,
        round(float(retrieved_row[PERCENTAGE_OUTPUT_COLUMN]), 6),
        bool(retrieved_row[FLAG_OUTPUT_COLUMN]),
        float(retrieved_row[THRESHOLD_OUTPUT_COLUMN]),
        str(retrieved_row[RUN_ID_OUTPUT_COLUMN]),
        str(retrieved_row[SCORED_AT_OUTPUT_COLUMN]),
        "PASS",
    ))

retrieval_df = spark.createDataFrame(
    retrieval_rows,
    schema=StructType([
        StructField("student_deidentified_hash", StringType(), False),
        StructField("attrition_risk_percentage", DoubleType(), True),
        StructField("attrition_risk_flag", BooleanType(), True),
        StructField("prediction_threshold", DoubleType(), True),
        StructField("mlflow_run_id", StringType(), True),
        StructField("scored_at", StringType(), True),
        StructField("retrieval_check", StringType(), False),
    ]),
)
display(retrieval_df)

print(f"Retrieval confirmed for {len(retrieval_rows)} students: each returns exactly one row with the "
      "correct percentage, flag, threshold, MLflow run id, and timestamp.")

### 29.5 Retrieval as the application would issue it (SQL)

In [0]:
_example_key = retrieval_sample_keys[0]
display(spark.sql(f"""
    SELECT student_deidentified_hash,
           attrition_risk_percentage,
           attrition_risk_flag,
           prediction_threshold,
           mlflow_run_id,
           scored_at
    FROM {PREDICTION_TABLE}
    WHERE student_deidentified_hash = '{_example_key}'
"""))

## 30. Phase 3 summary

In [0]:
print("=" * 96)
print("PHASE 3 SUMMARY - inference, risk flags, Delta persistence, retrieval validation")
print("=" * 96)
print(f"Inference population : {INFERENCE_POPULATION_DESCRIPTION}")
print(f"  rows               : {INFERENCE_POPULATION_ROWS:,}")
print(f"  distinct students  : {INFERENCE_POPULATION_STUDENTS:,}")
print(f"  known outcome      : {INFERENCE_KNOWN_OUTCOME_ROWS:,}")
print(f"  unknown outcome    : {INFERENCE_UNKNOWN_OUTCOME_ROWS:,}")
print()
print(f"Output rows          : {OUTPUT_ROWS:,} (one validated result per applicable student)")
print(f"Persisted rows       : {PERSISTED_ROWS:,}")
print(f"Persisted students   : {PERSISTED_STUDENTS:,}")
print()
print(f"Threshold            : {RISK_THRESHOLD} applied to the unrounded probability (>= true, < false)")
print(f"  flagged at risk    : {PERSISTED_FLAG_TRUE:,} ({100.0 * PERSISTED_FLAG_TRUE / PERSISTED_ROWS:.4f}%)")
print(f"  not flagged        : {PERSISTED_FLAG_FALSE:,} ({100.0 * PERSISTED_FLAG_FALSE / PERSISTED_ROWS:.4f}%)")
print(f"  probability range  : [{output_checks['min_probability']:.6f}, "
      f"{output_checks['max_probability']:.6f}], mean {output_checks['mean_probability']:.6f}")
print()
print(f"Validation           : {len(OUTPUT_VALIDATIONS)} pre-publish checks passed, "
      f"{len(PERSISTED_VALIDATIONS)} post-publish checks passed")
print("  no nulls, no duplicate students, probabilities in [0,1], percentages in [0,100],")
print("  Boolean flags only, threshold exactly 0.50, MLflow run id populated on every row")
print()
print(f"Delta table          : {PREDICTION_TABLE}")
print(f"  schema             : {', '.join(PREDICTION_OUTPUT_COLUMNS)}")
print("  write mode         : overwrite (latest snapshot)")
print("  source tables modified: none")
print(f"  staging table      : {STAGING_TABLE} (transient, dropped)")
print()
print("Compute              : the model is applied to the full population exactly once. Validation,")
print("                       publication, distribution, and retrieval all read Delta.")
print()
print(f"MLflow experiment    : {MLFLOW_EXPERIMENT_PATH}")
print(f"MLflow run id        : {MLFLOW_RUN_ID}")
print(f"Scored at            : {scored_at_timestamp.isoformat()}")
print()
print(f"Retrieval by {STUDENT_ID_COLUMN}: confirmed for {len(retrieval_rows)} students")
print("=" * 96)

In [0]:
display(spark.table("workspace.student_aggregate.student_attrition_risk_prediction").orderBy(F.rand()).limit(20))

## 31. Phase 3 human review outcome - approved 2026-08-06

The human reviewer approved Phase 3 and closed Sprint 2 machine-learning implementation.
Tasks **T22-T32** are marked complete in `tasks.md`. The approved output table
`workspace.student_aggregate.student_attrition_risk_prediction` was confirmed queryable with one
row per `student_deidentified_hash` and correct retrieval by the logical primary key.

No material refinement required approval in Phase 3.

### Routine corrections reported at this gate

| Correction | Reason |
|---|---|
| The scoring result is materialised once to a transient staging table, validated there, then published to the approved table and the staging table dropped | Serverless forbids `cache()`, so validating a lazy DataFrame and then writing it would run the eleven-join chain and all 100 trees twice over the full population. Staging applies the model once. Nothing reaches the approved table until every check passes, so the guarantee required by T27 is unchanged |
| The intermediate `attrition_risk_probability` lives in staging but is dropped at publication | The approved persisted schema is the six columns in `plan.md` section 10; adding a seventh needs approval and an SDD update. Staging is transient, so it can carry the column the validation needs |
| The flag is compared against the unrounded threshold rule as an explicit validation check | Proves the flag was never derived from the rounded percentage, so 0.50 is true and 0.49 is false |
| The inference population is taken from the Phase 1 assertions rather than recounted | Section 10 already measured and asserted the row count, uniqueness, and outcome breakdown on this exact DataFrame. Recounting would cost a full join pass to rediscover validated figures, and comparing the output against the Phase 1 figure is a stronger check than a self-consistent recount |
| Post-publish checks query the approved table, and the distribution and retrieval tests read Delta rather than rescoring | Verifies the table the application will actually read, at a fraction of the cost |
| Retrieval expectations are captured from staging before publication | Lets the retrieval test prove the staging-to-approved copy preserved every value exactly, without rescoring anything |
| Retrieval is tested on both a flagged and an unflagged student | Exercises both branches of the application lookup path |

### Compute profile

The eleven-join chain and the 100-tree forest are applied to the full population **once**, in
section 27. Every later step reads Delta. A naive ordering would have cost three full passes: one
to size the population, one to validate, and one to write.

### Known limitations accepted at final review

Three limitations recorded earlier continue to govern how the stored output should be read, and
are documented in full as L-1, L-2, and L-3 in `plan.md`:

1. **`attrition_risk_percentage` is a ranking score, not a calibrated likelihood.** Class weighting
   recentres probabilities on 0.50, so a stored value of 52 does not mean a 52% chance of attrition
   when the true base rate is 5.51%. Any application surface that presents this number to staff
   should describe it as a relative risk score.
2. **Discrimination is limited by the synthetic data, not by the model.** The target is drawn from a
   Beta distribution selected by a field-of-education value that is never written to the fact table,
   and the field of education the model can see is independent of it. The achievable ceiling is
   roughly 0.59 ROC-AUC even with perfect information.
3. **Scoring covers the whole modelling population, including rows used in training.** This follows
   the approved decision to score every valid student row. Training rows are scored optimistically,
   which affects the stored percentages but not the Phase 2 metrics, which were measured on the
   untouched test split.

### Sprint 2 closure

Sprint 2 machine-learning specification stories US-1 to US-8 are accepted. Parent Product Backlog
stories US-05, US-06, and US-07 remain the project-level acceptance gate outside this Sprint 2
specification set.